In [ ]:
# Notebook 09
# Target-Oriented Association Analysis

**Project:** Machine Learning-Based Prediction of Degradation in PEM Fuel Cells Using Time-Series Operational Data

This notebook investigates statistical associations between PEM fuel cell
operational variables, with particular emphasis on stack voltage as the selected
performance target.

The analysis evaluates linear, monotonic, and more general statistical
dependencies using Pearson correlation, Spearman rank correlation, and Mutual
Information.

Because the PEMFC is operated under a dynamic load cycle, strong associations
with voltage may reflect operating-condition effects, physical subsystem
coupling, mathematical relationships, or potentially durability-related
behaviour. Association strength is therefore interpreted carefully and is not
treated as direct evidence of degradation or causality.

The findings from this notebook will support subsequent degradation
representation, feature engineering, feature selection, and predictive modelling.

In [1]:
## Objectives

The objectives of this notebook are to:

- Investigate statistical associations between PEMFC operational variables and
  stack voltage.
- Quantify linear associations using Pearson correlation.
- Quantify monotonic associations using Spearman rank correlation.
- Detect more general statistical dependence using Mutual Information.
- Compare Pearson and Spearman results to identify association structures that
  are represented differently by linear and rank-based measures.
- Examine selected voltage relationships across durability stages.
- Identify strong predictor-to-predictor relationships relevant to redundancy
  and operating-protocol dependence.
- Distinguish load/control-driven, physically coupled, mathematical, and
  potentially ageing-related associations.
- Establish an evidence base for subsequent feature engineering and feature
  selection.

Association strength alone will not be used to automatically retain or remove
predictor variables.

Association analysis libraries imported successfully.


In [ ]:
## Notebook Workflow

This notebook follows the structure below:

### D3 Target-Oriented Association Analysis

- D3.1 Introduction and Objective
- D3.2 Association Analysis Preparation
- D3.3 Pearson Correlation Analysis
- D3.4 Spearman Rank Correlation Analysis
- D3.5 Pearson–Spearman Comparison
- D3.6 Mutual Information Analysis
- D3.7 Selected Stage-Wise Association Analysis
- D3.8 Integrated Association Assessment
- D3.9 Association Summary and Research Insights

In [2]:
# ============================================================
# Import Required Libraries
# ============================================================

import warnings
warnings.filterwarnings("default")

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from scipy.stats import spearmanr

from sklearn.feature_selection import mutual_info_regression

print("Required libraries imported successfully.")

Required libraries imported successfully.


In [3]:
# ============================================================
# Notebook Configuration
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

plt.style.use("default")

print("Notebook configured successfully.")

Notebook configured successfully.


In [4]:
# ============================================================
# Define Project Paths
# ============================================================

project_root = Path.cwd().parent

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"

figures_dir = project_root / "figures" / "association_analysis"
results_dir = project_root / "results" / "association_analysis"

figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Project Root      :", project_root)
print("Processed Data    :", processed_data_dir)
print("Association Figures:", figures_dir)
print("Association Results:", results_dir)

Project Root      : C:\Users\usman\Desktop\PEMFC_Dissertation
Processed Data    : C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed
Association Figures: C:\Users\usman\Desktop\PEMFC_Dissertation\figures\association_analysis
Association Results: C:\Users\usman\Desktop\PEMFC_Dissertation\results\association_analysis


In [5]:
# ============================================================
# Configure Project Source Package
# ============================================================

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root added to Python path successfully.")

Project root added to Python path successfully.


In [6]:
# ============================================================
# Load Processed Dataset
# ============================================================

dataset_path = processed_data_dir / "operational_cleaned.csv"

combined_df = pd.read_csv(dataset_path)

print("Dataset loaded successfully.")
print("Dataset Path:", dataset_path)

Dataset loaded successfully.
Dataset Path: C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed\operational_cleaned.csv


In [7]:
# ============================================================
# Verify Dataset
# ============================================================

print("=" * 60)
print("Dataset Information")
print("=" * 60)

print(f"Rows    : {combined_df.shape[0]:,}")
print(f"Columns : {combined_df.shape[1]}")

display(combined_df.head())

Dataset Information
Rows    : 3,629,680
Columns : 18


,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
0,50,1.7610,0.0000,0.9375,0.0000,109.9013,110.3273,109.8001,108.7921,83.1282,54.4647,71.9515,39.4532,64.4643,69.6736,56.3375,0.0700,0.2910
1,50,2.7610,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.8934,83.0788,54.4647,71.9021,39.3870,64.4390,69.6612,56.3249,0.0700,0.2910
2,50,3.7610,0.0000,0.9372,0.0000,110.3060,110.3273,109.8001,108.7921,83.0788,54.5293,71.8897,39.4135,64.4866,69.6859,56.2997,0.0700,0.2910
3,50,4.7610,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.7921,83.1035,54.4777,71.9391,39.3870,64.4390,69.6859,56.3249,0.0700,0.2910
4,50,5.7610,0.0000,0.9372,0.0000,109.9013,110.3273,109.6990,108.8934,83.0911,54.4777,71.8897,39.3738,64.4895,69.6612,56.2492,0.0700,0.2910


In [8]:
# ============================================================
# Dataset Structure
# ============================================================

combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3629680 entries, 0 to 3629679
Data columns (total 18 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   operating_hour               int64  
 1   time                         float64
 2   current                      float64
 3   voltage                      float64
 4   power                        float64
 5   pressure_anode_inlet         float64
 6   pressure_anode_outlet        float64
 7   pressure_cathode_inlet       float64
 8   pressure_cathode_outlet      float64
 9   temp_anode_endplate          float64
 10  temp_anode_dewpoint_water    float64
 11  temp_anode_inlet             float64
 12  temp_anode_outlet            float64
 13  temp_cathode_dewpoint_water  float64
 14  temp_cathode_inlet           float64
 15  temp_cathode_outlet          float64
 16  total_anode_stack_flow       float64
 17  total_cathode_stack_flow     float64
dtypes: float64(17), int64(1)
memory usage: 498.5 MB


In [9]:
# ============================================================
# Missing Values
# ============================================================

missing = combined_df.isnull().sum()
missing = missing[missing > 0]

print("=" * 60)
print("Missing Value Check")
print("=" * 60)

if missing.empty:
    print("No missing values found.")
else:
    display(missing)

Missing Value Check
No missing values found.


In [10]:
# ============================================================
# Duplicate Records
# ============================================================

duplicates = combined_df.duplicated().sum()

print("=" * 60)
print("Duplicate Record Check")
print("=" * 60)

print(f"Duplicate Rows : {duplicates:,}")

Duplicate Record Check
Duplicate Rows : 0


In [11]:
# ============================================================
# Dataset Variables
# ============================================================

print("=" * 60)
print("Dataset Variables")
print("=" * 60)

for i, column in enumerate(combined_df.columns, start=1):
    print(f"{i:2d}. {column}")

Dataset Variables
 1. operating_hour
 2. time
 3. current
 4. voltage
 5. power
 6. pressure_anode_inlet
 7. pressure_anode_outlet
 8. pressure_cathode_inlet
 9. pressure_cathode_outlet
10. temp_anode_endplate
11. temp_anode_dewpoint_water
12. temp_anode_inlet
13. temp_anode_outlet
14. temp_cathode_dewpoint_water
15. temp_cathode_inlet
16. temp_cathode_outlet
17. total_anode_stack_flow
18. total_cathode_stack_flow


In [12]:
# ============================================================
# D3.2.1 Define Variable Groups
# ============================================================

TARGET_VARIABLE = "voltage"
STAGE_VARIABLE = "operating_hour"

electrical_variables = [
    "current",
    "voltage",
    "power",
]

pressure_variables = [
    "pressure_anode_inlet",
    "pressure_anode_outlet",
    "pressure_cathode_inlet",
    "pressure_cathode_outlet",
]

temperature_variables = [
    "temp_anode_endplate",
    "temp_anode_dewpoint_water",
    "temp_anode_inlet",
    "temp_anode_outlet",
    "temp_cathode_dewpoint_water",
    "temp_cathode_inlet",
    "temp_cathode_outlet",
]

reactant_flow_variables = [
    "total_anode_stack_flow",
    "total_cathode_stack_flow",
]

association_variables = (
    electrical_variables
    + pressure_variables
    + temperature_variables
    + reactant_flow_variables
)

print("Association variable groups defined successfully.")
print(f"Target variable      : {TARGET_VARIABLE}")
print(f"Stage variable       : {STAGE_VARIABLE}")
print(f"Measured variables   : {len(association_variables)}")

Association variable groups defined successfully.
Target variable      : voltage
Stage variable       : operating_hour
Measured variables   : 16


In [13]:
# ============================================================
# D3.2.2 Define Candidate Predictor Variables
# ============================================================

candidate_predictors = [
    variable
    for variable in association_variables
    if variable not in [TARGET_VARIABLE, "power"]
]

print("Candidate predictor variables defined successfully.")
print(f"Number of candidate predictors: {len(candidate_predictors)}")

for variable in candidate_predictors:
    print(f"- {variable}")

Candidate predictor variables defined successfully.
Number of candidate predictors: 14
- current
- pressure_anode_inlet
- pressure_anode_outlet
- pressure_cathode_inlet
- pressure_cathode_outlet
- temp_anode_endplate
- temp_anode_dewpoint_water
- temp_anode_inlet
- temp_anode_outlet
- temp_cathode_dewpoint_water
- temp_cathode_inlet
- temp_cathode_outlet
- total_anode_stack_flow
- total_cathode_stack_flow


In [14]:
# ============================================================
# D3.2.3 Verify Required Columns
# ============================================================

required_columns = [
    STAGE_VARIABLE,
    *association_variables,
]

missing_columns = [
    column
    for column in required_columns
    if column not in combined_df.columns
]

if missing_columns:
    raise KeyError(
        "The following required columns are missing from the dataset: "
        f"{missing_columns}"
    )

print("All required association-analysis columns are available.")

All required association-analysis columns are available.


In [15]:
# ============================================================
# D3.2.4 Create Association Analysis Dataset
# ============================================================

association_df = combined_df[
    [
        STAGE_VARIABLE,
        *association_variables,
    ]
].copy()

association_df = (
    association_df
    .sort_values(by=[STAGE_VARIABLE])
    .reset_index(drop=True)
)

print("Association analysis dataset created successfully.")
print(f"Rows    : {association_df.shape[0]:,}")
print(f"Columns : {association_df.shape[1]}")

Association analysis dataset created successfully.
Rows    : 3,629,680
Columns : 17


In [16]:
# ============================================================
# D3.2.5 Preview Association Dataset
# ============================================================

display(association_df.head())

,operating_hour,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
0,50,0.0000,0.9375,0.0000,109.9013,110.3273,109.8001,108.7921,83.1282,54.4647,71.9515,39.4532,64.4643,69.6736,56.3375,0.0700,0.2910
1,50,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.8934,83.0788,54.4647,71.9021,39.3870,64.4390,69.6612,56.3249,0.0700,0.2910
2,50,0.0000,0.9372,0.0000,110.3060,110.3273,109.8001,108.7921,83.0788,54.5293,71.8897,39.4135,64.4866,69.6859,56.2997,0.0700,0.2910
3,50,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.7921,83.1035,54.4777,71.9391,39.3870,64.4390,69.6859,56.3249,0.0700,0.2910
4,50,0.0000,0.9372,0.0000,109.9013,110.3273,109.6990,108.8934,83.0911,54.4777,71.8897,39.3738,64.4895,69.6612,56.2492,0.0700,0.2910


In [17]:
# ============================================================
# D3.2.6 Verify Numeric Data Types
# ============================================================

non_numeric_variables = [
    variable
    for variable in required_columns
    if not pd.api.types.is_numeric_dtype(association_df[variable])
]

if non_numeric_variables:
    print(
        "Non-numeric variables detected:",
        non_numeric_variables,
    )
else:
    print("All association-analysis variables are numeric.")

All association-analysis variables are numeric.


In [18]:
# ============================================================
# D3.2.7 Verify Missing and Infinite Values
# ============================================================

missing_values = association_df.isna().sum()
missing_values = missing_values[missing_values > 0]

infinite_counts = pd.Series(
    {
        column: np.isinf(
            association_df[column].to_numpy()
        ).sum()
        for column in required_columns
    }
)

infinite_counts = infinite_counts[
    infinite_counts > 0
]

print("=" * 60)
print("Association Dataset Integrity Check")
print("=" * 60)

if missing_values.empty:
    print("Missing values  : None")
else:
    print("\nMissing values:")
    display(missing_values)

if infinite_counts.empty:
    print("Infinite values : None")
else:
    print("\nInfinite values:")
    display(infinite_counts)

Association Dataset Integrity Check
Missing values  : None
Infinite values : None


In [19]:
# ============================================================
# D3.2.8 Verify Durability Stages
# ============================================================

available_stages = sorted(
    association_df[STAGE_VARIABLE].unique()
)

print("=" * 60)
print("Available Durability Stages")
print("=" * 60)

print(available_stages)
print(f"\nNumber of stages: {len(available_stages)}")

Available Durability Stages
[np.int64(50), np.int64(100), np.int64(150), np.int64(200), np.int64(250), np.int64(300), np.int64(350), np.int64(400), np.int64(450), np.int64(500), np.int64(550), np.int64(600), np.int64(650), np.int64(700), np.int64(750), np.int64(800), np.int64(850), np.int64(900), np.int64(950), np.int64(1000)]

Number of stages: 20


In [20]:
# ============================================================
# D3.2.9 Association Analysis Preparation Summary
# ============================================================

preparation_summary = pd.DataFrame(
    {
        "Item": [
            "Target Variable",
            "Association Variables",
            "Candidate Predictors",
            "Durability Stages",
            "Analysis Rows",
            "Missing Values",
            "Infinite Values",
        ],
        "Value": [
            TARGET_VARIABLE,
            len(association_variables),
            len(candidate_predictors),
            association_df[STAGE_VARIABLE].nunique(),
            f"{len(association_df):,}",
            int(association_df.isna().sum().sum()),
            int(
                sum(
                    np.isinf(
                        association_df[column].to_numpy()
                    ).sum()
                    for column in required_columns
                )
            ),
        ],
    }
)

display(preparation_summary)

,Item,Value
0,Target Variable,voltage
1,Association Variables,16
2,Candidate Predictors,14
3,Durability Stages,20
4,Analysis Rows,"3,629,680"
5,Missing Values,0
6,Infinite Values,0


In [ ]:
## D3.3 Pearson Correlation Analysis

Pearson correlation is used to examine the strength and direction of linear
relationships between PEMFC variables.

The analysis first examines the overall correlation structure and then focuses
specifically on associations between the operational variables and voltage.

In [21]:
# ============================================================
# D3.3.2 Calculate Global Pearson Correlation Matrix
# ============================================================

pearson_matrix = association_df[
    association_variables
].corr(method="pearson")

print("=" * 60)
print("Pearson Correlation Matrix")
print("=" * 60)

display(pearson_matrix)

Pearson Correlation Matrix


,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
current,1.0000,-0.9702,0.9896,0.1083,-0.1108,0.6092,-0.4485,0.3856,-0.0084,0.4498,0.0177,-0.0134,0.7999,0.3182,0.9803,0.9805
voltage,-0.9702,1.0000,-0.9756,-0.1128,0.0892,-0.6703,0.3024,-0.4041,-0.0199,-0.3859,-0.0399,-0.0146,-0.7423,-0.2414,-0.9218,-0.9222
power,0.9896,-0.9756,1.0000,0.1133,-0.0970,0.6318,-0.3784,0.3533,-0.0150,0.4101,0.0006,-0.0130,0.7958,0.2762,0.9498,0.9501
pressure_anode_inlet,0.1083,-0.1128,0.1133,1.0000,0.9441,0.1683,0.0573,0.0558,0.0002,0.0128,0.0648,0.0123,0.0551,0.0104,0.1048,0.1048
pressure_anode_outlet,-0.1108,0.0892,-0.0970,0.9441,1.0000,0.0485,0.1905,-0.0003,-0.0202,-0.1075,-0.0906,-0.0107,-0.1280,-0.0464,-0.1198,-0.1196
pressure_cathode_inlet,0.6092,-0.6703,0.6318,0.1683,0.0485,1.0000,0.3742,0.1411,-0.0007,0.0353,-0.0641,0.0487,0.3920,-0.2285,0.5688,0.5691
pressure_cathode_outlet,-0.4485,0.3024,-0.3784,0.0573,0.1905,0.3742,1.0000,-0.3050,0.0120,-0.5148,-0.1080,0.0735,-0.5015,-0.6457,-0.5256,-0.5250
temp_anode_endplate,0.3856,-0.4041,0.3533,0.0558,-0.0003,0.1411,-0.3050,1.0000,-0.0226,0.2364,0.3085,-0.0375,0.3582,0.4099,0.4085,0.4084
temp_anode_dewpoint_water,-0.0084,-0.0199,-0.0150,0.0002,-0.0202,-0.0007,0.0120,-0.0226,1.0000,0.1841,0.1539,0.5829,0.0517,-0.0588,-0.0088,-0.0088
temp_anode_inlet,0.4498,-0.3859,0.4101,0.0128,-0.1075,0.0353,-0.5148,0.2364,0.1841,1.0000,0.1014,0.1377,0.4090,0.4229,0.4834,0.4831


In [22]:
# ============================================================
# D3.3.3 Visualise Global Pearson Correlation Matrix
# ============================================================

# Suppress Seaborn internal PendingDeprecationWarning
warnings.filterwarnings(
    "ignore",
    category=PendingDeprecationWarning,
    module="seaborn.*"
)

fig, ax = plt.subplots(figsize=(14, 11))

sns.heatmap(
    pearson_matrix,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.4,
    cbar_kws={
        "label": "Pearson Correlation (r)",
        "shrink": 0.85
    },
    ax=ax
)

# Move horizontal variable names to the top
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")

ax.tick_params(
    axis="x",
    top=True,
    bottom=False,
    labeltop=True,
    labelbottom=False
)

# Improve readability of variable names
plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="left",
    rotation_mode="anchor"
)

plt.setp(
    ax.get_yticklabels(),
    rotation=0
)

ax.set_title(
    "Global Pearson Correlation Structure of PEMFC Variables",
    fontsize=14,
    pad=20
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()
plt.show()

<Figure size 1400x1100 with 2 Axes>

In [23]:
# ============================================================
# D3.3.4 Extract Voltage-Specific Pearson Associations
# ============================================================

voltage_pearson = (
    pearson_matrix[TARGET_VARIABLE]
    .drop(TARGET_VARIABLE)
    .rename("pearson_r")
    .to_frame()
)

voltage_pearson["abs_pearson_r"] = (
    voltage_pearson["pearson_r"].abs()
)

voltage_pearson = voltage_pearson.sort_values(
    "abs_pearson_r",
    ascending=False
)

print("=" * 60)
print("Pearson Associations with Voltage")
print("=" * 60)

display(voltage_pearson)

Pearson Associations with Voltage


,pearson_r,abs_pearson_r
power,-0.9756,0.9756
current,-0.9702,0.9702
total_cathode_stack_flow,-0.9222,0.9222
total_anode_stack_flow,-0.9218,0.9218
temp_cathode_inlet,-0.7423,0.7423
pressure_cathode_inlet,-0.6703,0.6703
temp_anode_endplate,-0.4041,0.4041
temp_anode_inlet,-0.3859,0.3859
pressure_cathode_outlet,0.3024,0.3024
temp_cathode_outlet,-0.2414,0.2414


In [25]:
# ============================================================
# D3.3.5 Pearson Associations:
# Candidate Predictors vs Voltage
# Removing Power
# ============================================================

candidate_voltage_pearson = (
    voltage_pearson
    .loc[candidate_predictors]
    .sort_values(
        "abs_pearson_r",
        ascending=False
    )
)

print("=" * 60)
print("Candidate Predictor Associations with Voltage")
print("=" * 60)

display(predictor_pearson)

Candidate Predictor Associations with Voltage


NameError: name 'predictor_pearson' is not defined

In [26]:
# ============================================================
# D3.3.6 Visualise Pearson Associations with Voltage
# ============================================================

plot_data = candidate_voltage_pearson.sort_values(
    "pearson_r"
)

# Red = negative association
# Blue = positive association
bar_colors = [
    "red" if value < 0 else "blue"
    for value in plot_data["pearson_r"]
]

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(
    plot_data.index,
    plot_data["pearson_r"],
    color=bar_colors,
    alpha=0.8,
    edgecolor="black",
    linewidth=0.6
)

# Zero-reference line
ax.axvline(
    0,
    color="black",
    linewidth=1.1,
    linestyle="--"
)

# Add correlation coefficient labels
for bar, value in zip(
    bars,
    plot_data["pearson_r"]
):
    
    y_position = bar.get_y() + bar.get_height() / 2

    if value < 0:
        ax.text(
            value - 0.025,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=9,
            fontweight="bold"
        )
    else:
        ax.text(
            value + 0.025,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=9,
            fontweight="bold"
        )

ax.set_xlabel(
    "Pearson Correlation with Voltage (r)",
    fontsize=11
)

ax.set_ylabel(
    "Operational Variable",
    fontsize=11
)

ax.set_title(
    "Linear Associations Between Operational Variables and Voltage",
    fontsize=13,
    pad=12
)

# Extra space allows coefficient labels to remain visible
ax.set_xlim(-1.10, 1.10)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()
plt.show()

<Figure size 1000x700 with 1 Axes>

In [ ]:
### D3.3.7 Visual Assessment of Predictor–Voltage Relationships

Selected predictor–voltage relationships are visually examined to assess their structure and the suitability of linear Pearson interpretation.

In [27]:
# ============================================================
# D3.3.7 Prepare Stage-Stratified Visualisation Sample
# ============================================================

# Selected predictors representing different association patterns
visual_predictors = [
    "current",
    "total_cathode_stack_flow",
    "pressure_cathode_inlet",
    "temp_cathode_inlet",
    "temp_anode_endplate",
    "pressure_anode_inlet"
]

# ------------------------------------------------------------
# Define durability-stage sampling
# ------------------------------------------------------------

stage_column = "operating_hour"

durability_stages = sorted(
    association_df[stage_column].unique()
)

target_visual_sample_size = 50_000

samples_per_stage = (
    target_visual_sample_size // len(durability_stages)
)

# ------------------------------------------------------------
# Create stage-stratified random sample
# ------------------------------------------------------------

visual_columns = (
    [stage_column, TARGET_VARIABLE]
    + visual_predictors
)

visual_sample = (
    association_df[visual_columns]
    .groupby(
        stage_column,
        group_keys=False
    )
    .sample(
        n=samples_per_stage,
        random_state=42
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Existing full-dataset Pearson coefficients
# ------------------------------------------------------------

visual_pearson_summary = pd.DataFrame({
    "predictor": visual_predictors,
    "pearson_r": [
        voltage_pearson.loc[var, "pearson_r"]
        for var in visual_predictors
    ]
})

# ------------------------------------------------------------
# Check stage representation
# ------------------------------------------------------------

stage_sample_counts = (
    visual_sample[stage_column]
    .value_counts()
    .sort_index()
    .rename("sampled_observations")
    .to_frame()
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("=" * 70)
print("Stage-Stratified Visual Relationship Assessment")
print("=" * 70)

print(
    f"Full observations used for Pearson statistics: "
    f"{len(association_df):,}"
)

print(
    f"Number of durability stages: "
    f"{len(durability_stages)}"
)

print(
    f"Observations sampled per stage: "
    f"{samples_per_stage:,}"
)

print(
    f"Total observations sampled for visualisation: "
    f"{len(visual_sample):,}"
)

print(
    f"Predictors selected for visual assessment: "
    f"{len(visual_predictors)}"
)

print("\nPearson coefficients from full dataset:")

display(
    visual_pearson_summary.round(4)
)

print("\nSample representation across durability stages:")

display(
    stage_sample_counts
)

Stage-Stratified Visual Relationship Assessment
Full observations used for Pearson statistics: 3,629,680
Number of durability stages: 20
Observations sampled per stage: 2,500
Total observations sampled for visualisation: 50,000
Predictors selected for visual assessment: 6

Pearson coefficients from full dataset:


,predictor,pearson_r
0,current,-0.9702
1,total_cathode_stack_flow,-0.9222
2,pressure_cathode_inlet,-0.6703
3,temp_cathode_inlet,-0.7423
4,temp_anode_endplate,-0.4041
5,pressure_anode_inlet,-0.1128



Sample representation across durability stages:


,sampled_observations
operating_hour,
50,2500
100,2500
150,2500
200,2500
250,2500
300,2500
350,2500
400,2500
450,2500


In [ ]:
### D3.3.8 Visual Assessment of Linearity

Visualise selected predictor–voltage relationships to assess whether their structure is reasonably linear and to support interpretation of the Pearson coefficients.

In [28]:
# ============================================================
# D3.3.8 Visual Assessment of Linearity
# ============================================================

# Selected predictors defined in D3.3.7
predictors_to_visualise = [
    "current",
    "total_cathode_stack_flow",
    "pressure_cathode_inlet",
    "temp_cathode_inlet",
    "temp_anode_endplate",
    "pressure_anode_inlet"
]

# ------------------------------------------------------------
# Create scatter plots
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(14, 15)
)

axes = axes.flatten()

for ax, predictor in zip(axes, predictors_to_visualise):

    # Pearson coefficient calculated from the FULL dataset
    r_value = voltage_pearson.loc[predictor, "pearson_r"]

    # Scatter plot uses only the balanced visual sample
    ax.scatter(
        visual_sample[predictor],
        visual_sample[TARGET_VARIABLE],
        s=5,
        alpha=0.15
    )

    # Linear trend line fitted to the visual sample
    sns.regplot(
        data=visual_sample,
        x=predictor,
        y=TARGET_VARIABLE,
        scatter=False,
        ci=None,
        line_kws={"linewidth": 2},
        ax=ax
    )

    ax.set_title(
        f"{predictor}\nPearson r = {r_value:.3f}",
        fontsize=11
    )

    ax.set_xlabel(predictor)
    ax.set_ylabel("Voltage (V)")

    ax.grid(
        linestyle="--",
        alpha=0.3
    )

plt.suptitle(
    "Visual Assessment of Predictor–Voltage Linearity",
    fontsize=15,
    y=1.01
)

plt.tight_layout()
plt.show()

<Figure size 1400x1500 with 6 Axes>

In [29]:
# ============================================================
# D3.3.7 Identify Strong Predictor-to-Predictor Associations
# Threshold |x|>=0.70
# ============================================================

predictor_matrix = pearson_matrix.loc[
    candidate_predictors,
    candidate_predictors
]

upper_triangle = np.triu(
    np.ones(predictor_matrix.shape),
    k=1
).astype(bool)

strong_predictor_pairs = (
    predictor_matrix
    .where(upper_triangle)
    .stack()
    .reset_index()
)

strong_predictor_pairs.columns = [
    "variable_1",
    "variable_2",
    "pearson_r"
]

strong_predictor_pairs["abs_pearson_r"] = (
    strong_predictor_pairs["pearson_r"].abs()
)

strong_predictor_pairs = (
    strong_predictor_pairs[
        strong_predictor_pairs["abs_pearson_r"] >= 0.70
    ]
    .sort_values(
        "abs_pearson_r",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("Strong Predictor-to-Predictor Pearson Associations")
print("=" * 60)

display(strong_predictor_pairs)

Strong Predictor-to-Predictor Pearson Associations


,variable_1,variable_2,pearson_r,abs_pearson_r
0,total_anode_stack_flow,total_cathode_stack_flow,1.0000,1.0000
1,current,total_cathode_stack_flow,0.9805,0.9805
2,current,total_anode_stack_flow,0.9803,0.9803
3,pressure_anode_inlet,pressure_anode_outlet,0.9441,0.9441
4,temp_cathode_inlet,total_cathode_stack_flow,0.8099,0.8099
5,temp_cathode_inlet,total_anode_stack_flow,0.8096,0.8096
6,current,temp_cathode_inlet,0.7999,0.7999


In [ ]:
### D3.4 Spearman Rank Correlation Analysis

Spearman correlation is used to evaluate monotonic associations between voltage and the operational variables without requiring the relationships to be linear.

In [30]:
# ============================================================
# Calculate Spearman Correlation Matrix
# ============================================================

spearman_corr = association_df[association_variables].corr(method="spearman")

print("=" * 60)
print("Spearman Correlation Matrix")
print("=" * 60)

display(spearman_corr.round(4))

Spearman Correlation Matrix


,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
current,1.0000,-0.9284,0.9534,0.3730,-0.3048,0.7619,-0.2678,0.3423,-0.0254,0.3874,0.0165,-0.0019,0.6942,0.2075,0.9165,0.9167
voltage,-0.9284,1.0000,-0.8612,-0.3904,0.2762,-0.7612,0.2239,-0.4528,0.0006,-0.3709,-0.0611,-0.0214,-0.6558,-0.2041,-0.9281,-0.9276
power,0.9534,-0.8612,1.0000,0.3600,-0.3026,0.7476,-0.2841,0.2486,-0.0554,0.3844,-0.0528,-0.0319,0.6986,0.1882,0.8873,0.8886
pressure_anode_inlet,0.3730,-0.3904,0.3600,1.0000,0.2599,0.5161,0.1930,0.0865,-0.0013,0.0698,-0.0028,0.0268,0.1626,-0.0258,0.3875,0.3868
pressure_anode_outlet,-0.3048,0.2762,-0.3026,0.2599,1.0000,-0.1071,0.4737,-0.1147,-0.0600,-0.3122,-0.4309,-0.0484,-0.3858,-0.1539,-0.3092,-0.3089
pressure_cathode_inlet,0.7619,-0.7612,0.7476,0.5161,-0.1071,1.0000,0.1906,0.2478,-0.0176,0.1481,-0.0349,0.0328,0.5655,-0.0659,0.7806,0.7814
pressure_cathode_outlet,-0.2678,0.2239,-0.2841,0.1930,0.4737,0.1906,1.0000,-0.2519,0.0307,-0.5349,-0.0941,0.0897,-0.4089,-0.5943,-0.3010,-0.3019
temp_anode_endplate,0.3423,-0.4528,0.2486,0.0865,-0.1147,0.2478,-0.2519,1.0000,-0.0040,0.2625,0.3248,-0.0107,0.3537,0.3840,0.4058,0.4064
temp_anode_dewpoint_water,-0.0254,0.0006,-0.0554,-0.0013,-0.0600,-0.0176,0.0307,-0.0040,1.0000,0.0650,0.2168,0.4523,-0.0104,0.0022,-0.0386,-0.0401
temp_anode_inlet,0.3874,-0.3709,0.3844,0.0698,-0.3122,0.1481,-0.5349,0.2625,0.0650,1.0000,0.1038,0.0421,0.3594,0.5336,0.4204,0.4210


In [ ]:
#### Global Spearman Correlation Structure

The heatmap visualises the direction and strength of monotonic associations among the PEMFC operational variables.

In [31]:
# ============================================================
# Visualise Global Spearman Correlation Matrix
# ============================================================

plt.figure(figsize=(14, 11))

ax = sns.heatmap(
    spearman_corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    annot=False,
    linewidths=0.3,
    cbar_kws={"label": "Spearman Correlation (ρ)"}
)

# Move x-axis labels to the top
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")

plt.xticks(rotation=45, ha="left")
plt.yticks(rotation=0)

plt.title(
    "Global Spearman Correlation Structure of PEMFC Variables",
    fontsize=14,
    pad=20
)

plt.tight_layout()
plt.show()

<Figure size 1400x1100 with 2 Axes>

In [ ]:
#### Spearman Associations with Voltage

Spearman coefficients are extracted to identify operational variables showing the strongest monotonic associations with voltage.

In [32]:
# ============================================================
# Extract Spearman Associations with Voltage
# ============================================================

voltage_spearman = (
    spearman_corr["voltage"]
    .drop("voltage")
    .to_frame(name="spearman_rho")
)

voltage_spearman["abs_spearman_rho"] = (
    voltage_spearman["spearman_rho"].abs()
)

voltage_spearman = voltage_spearman.sort_values(
    "abs_spearman_rho",
    ascending=False
)

print("=" * 60)
print("Spearman Associations with Voltage")
print("=" * 60)

display(voltage_spearman.round(4))

Spearman Associations with Voltage


,spearman_rho,abs_spearman_rho
current,-0.9284,0.9284
total_anode_stack_flow,-0.9281,0.9281
total_cathode_stack_flow,-0.9276,0.9276
power,-0.8612,0.8612
pressure_cathode_inlet,-0.7612,0.7612
temp_cathode_inlet,-0.6558,0.6558
temp_anode_endplate,-0.4528,0.4528
pressure_anode_inlet,-0.3904,0.3904
temp_anode_inlet,-0.3709,0.3709
pressure_anode_outlet,0.2762,0.2762


In [ ]:
#### Candidate Predictor Associations

Power is excluded from the candidate predictor ranking because it is mathematically derived from voltage and current.

In [33]:
# ============================================================
# Candidate Predictor Spearman Associations with Voltage
# ============================================================

candidate_voltage_spearman = voltage_spearman.drop(
    index="power",
    errors="ignore"
)

print("=" * 60)
print("Candidate Predictor Spearman Associations with Voltage")
print("=" * 60)

display(candidate_voltage_spearman.round(4))

Candidate Predictor Spearman Associations with Voltage


,spearman_rho,abs_spearman_rho
current,-0.9284,0.9284
total_anode_stack_flow,-0.9281,0.9281
total_cathode_stack_flow,-0.9276,0.9276
pressure_cathode_inlet,-0.7612,0.7612
temp_cathode_inlet,-0.6558,0.6558
temp_anode_endplate,-0.4528,0.4528
pressure_anode_inlet,-0.3904,0.3904
temp_anode_inlet,-0.3709,0.3709
pressure_anode_outlet,0.2762,0.2762
pressure_cathode_outlet,0.2239,0.2239


In [34]:
# ============================================================
# Visualise Spearman Associations with Voltage
# ============================================================

plot_data = candidate_voltage_spearman.sort_values(
    "spearman_rho",
    ascending=True
)

values = plot_data["spearman_rho"]

colors = [
    "red" if value < 0 else "blue"
    for value in values
]

fig, ax = plt.subplots(figsize=(10, 8))

bars = ax.barh(
    plot_data.index,
    values,
    color=colors,
    alpha=0.8,
    edgecolor="black",
    linewidth=0.5
)

# Zero reference line
ax.axvline(
    0,
    color="black",
    linewidth=1
)

# Add coefficient labels
for bar, value in zip(bars, values):

    y_position = bar.get_y() + bar.get_height() / 2

    if value < 0:
        ax.text(
            value - 0.02,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=9,
            fontweight="bold"
        )
    else:
        ax.text(
            value + 0.02,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=9,
            fontweight="bold"
        )

ax.set_xlim(-1.08, 1.08)

ax.set_xlabel("Spearman Correlation with Voltage (ρ)")
ax.set_ylabel("Operational Variable")

ax.set_title(
    "Monotonic Associations Between Operational Variables and Voltage"
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()
plt.show()

<Figure size 1000x800 with 1 Axes>

In [ ]:
### D3.7.1 Stage-Wise Spearman Associations with Voltage

To assess whether predictor–voltage relationships remain stable as the PEMFC progresses through durability stages, Spearman rank correlation is calculated separately at each operating hour for every candidate predictor.

This analysis evaluates whether the direction and strength of monotonic associations with voltage remain consistent across ageing stages.

Stable stage-wise associations suggest that the underlying predictor–voltage relationship is broadly preserved despite changes in absolute voltage level. In contrast, substantial changes in correlation strength or sign may indicate durability-dependent changes in the relationship between a predictor and voltage.

The analysis includes all candidate operational predictors and excludes:

- `voltage`, because it is the target variable;
- `power`, because it is mathematically derived from voltage and current.

The resulting stage-wise association table and heatmap are used to evaluate temporal stability of predictor–voltage relationships across the complete durability experiment.

In [35]:
# ============================================================
# D3.7.1 Stage-Wise Spearman Associations with Voltage
# ============================================================

stagewise_spearman_results = {}

for stage, stage_df in association_df.groupby(
    STAGE_VARIABLE,
    sort=True
):

    stage_correlations = (
        stage_df[
            candidate_predictors + [TARGET_VARIABLE]
        ]
        .corr(method="spearman")[TARGET_VARIABLE]
        .drop(TARGET_VARIABLE)
    )

    stagewise_spearman_results[stage] = stage_correlations


stagewise_spearman_table = pd.DataFrame(
    stagewise_spearman_results
)

stagewise_spearman_table.index.name = "predictor"
stagewise_spearman_table.columns.name = "operating_hour"

print("=" * 70)
print("Stage-Wise Spearman Correlations with Voltage")
print("=" * 70)

display(stagewise_spearman_table.round(4))

Stage-Wise Spearman Correlations with Voltage


operating_hour,50,100,150,200,250,300,350,400,450,500,550,600,650,700,750,800,850,900,950,1000
predictor,,,,,,,,,,,,,,,,,,,,
current,-0.9346,-0.9375,-0.9394,-0.9339,-0.9307,-0.9298,-0.9309,-0.9268,-0.9354,-0.9366,-0.9341,-0.9485,-0.9188,-0.9400,-0.9388,-0.9505,-0.9408,-0.9403,-0.9414,-0.9349
pressure_anode_inlet,-0.4804,-0.4927,-0.4946,-0.4305,-0.3945,-0.4077,-0.3798,-0.4145,-0.3755,-0.4526,-0.4361,-0.4607,-0.4410,-0.4131,-0.4222,-0.2792,-0.3507,-0.3228,-0.2682,-0.3298
pressure_anode_outlet,0.5328,0.3955,0.4848,0.3393,0.3977,0.3583,0.2227,0.2993,0.2786,0.4801,0.5102,0.4472,0.4410,0.3890,0.4921,0.0368,0.2922,0.2574,0.3293,0.3557
pressure_cathode_inlet,-0.7680,-0.7881,-0.7948,-0.7799,-0.7943,-0.7825,-0.7726,-0.7604,-0.7617,-0.7617,-0.7601,-0.7734,-0.7666,-0.7763,-0.7834,-0.7397,-0.7795,-0.7886,-0.7744,-0.7191
pressure_cathode_outlet,0.2371,0.2282,0.1780,0.2064,0.1671,0.1698,0.2318,0.2188,0.2338,0.2452,0.2431,0.2367,0.2119,0.1987,0.2004,0.2422,0.1853,0.1774,0.2086,0.3735
temp_anode_endplate,-0.6210,-0.5436,-0.6190,-0.5760,-0.5817,-0.5435,-0.5644,-0.4348,-0.6016,-0.5873,-0.6545,-0.6866,-0.4778,-0.6049,-0.6347,-0.6579,-0.5771,-0.6102,-0.6053,-0.5825
temp_anode_dewpoint_water,-0.0252,-0.0041,-0.0417,-0.0299,-0.0310,-0.0193,-0.0163,-0.0377,-0.0025,-0.0076,-0.0357,-0.0358,-0.0532,-0.0324,-0.0160,-0.0335,-0.0151,-0.0088,-0.0393,-0.0294
temp_anode_inlet,-0.4641,-0.3544,-0.3153,-0.4034,-0.3403,-0.3430,-0.3109,-0.3609,-0.3571,-0.3955,-0.3804,-0.3851,-0.4380,-0.3537,-0.3580,-0.3738,-0.3524,-0.3728,-0.3278,-0.3625
temp_anode_outlet,-0.0543,0.0819,-0.0443,-0.0592,0.0467,-0.0045,0.1118,-0.0293,-0.0234,-0.1034,-0.0447,-0.0968,-0.1372,-0.0184,-0.0212,-0.1095,0.0581,0.0555,0.1094,-0.1172


In [37]:
# ============================================================
# D3.7.2 Visualise Stage-Wise Spearman Associations
# ============================================================

plt.figure(figsize=(18, 10))

ax = sns.heatmap(
    stagewise_spearman_table,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={
        "label": "Spearman Correlation with Voltage (ρ)"
    }
)

# Move operating-hour labels to the top
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")

ax.tick_params(
    axis="x",
    top=True,
    bottom=False,
    labeltop=True,
    labelbottom=False
)

plt.title(
    "Stage-Wise Spearman Associations with Voltage",
    fontsize=14,
    pad=45
)

plt.xlabel(
    "Durability Stage (Operating Hour)",
    labelpad=12
)

plt.ylabel("Candidate Predictor")

plt.tight_layout()

plt.show()

<Figure size 1800x1000 with 2 Axes>

In [ ]:
### D3.5 Pearson–Spearman Comparison

Pearson and Spearman associations are compared to assess whether predictor–voltage relationships are predominantly linear or show evidence of non-linear monotonic behaviour.

In [60]:
# ============================================================
# Create Pearson-Spearman Comparison Table
# comparing ∣ρ∣−∣r∣ not ρ−r because we are acquiring relationship strength not direction
# ============================================================

pearson_spearman_comparison = pd.DataFrame({
    "pearson_r": candidate_voltage_pearson["pearson_r"],
    "spearman_rho": candidate_voltage_spearman["spearman_rho"]
})

# Absolute strengths
pearson_spearman_comparison["abs_pearson"] = (
    pearson_spearman_comparison["pearson_r"].abs()
)

pearson_spearman_comparison["abs_spearman"] = (
    pearson_spearman_comparison["spearman_rho"].abs()
)

# Difference in association strength
pearson_spearman_comparison["strength_difference"] = (
    pearson_spearman_comparison["abs_spearman"]
    - pearson_spearman_comparison["abs_pearson"]
)

# Absolute difference for ranking
pearson_spearman_comparison["abs_strength_difference"] = (
    pearson_spearman_comparison["strength_difference"].abs()
)

# Sort by largest Pearson-Spearman difference
pearson_spearman_comparison = pearson_spearman_comparison.sort_values(
    "abs_strength_difference",
    ascending=False
)

print("=" * 65)
print("Pearson-Spearman Comparison for Voltage Predictors")
print("=" * 65)

display(pearson_spearman_comparison.round(4))

Pearson-Spearman Comparison for Voltage Predictors


,pearson_r,spearman_rho,abs_pearson,abs_spearman,strength_difference,abs_strength_difference
pressure_anode_inlet,-0.1128,-0.3904,0.1128,0.3904,0.2776,0.2776
pressure_anode_outlet,0.0892,0.2762,0.0892,0.2762,0.1870,0.1870
pressure_cathode_inlet,-0.6703,-0.7612,0.6703,0.7612,0.0909,0.0909
temp_cathode_inlet,-0.7423,-0.6558,0.7423,0.6558,-0.0864,0.0864
pressure_cathode_outlet,0.3024,0.2239,0.3024,0.2239,-0.0786,0.0786
temp_anode_endplate,-0.4041,-0.4528,0.4041,0.4528,0.0487,0.0487
current,-0.9702,-0.9284,0.9702,0.9284,-0.0417,0.0417
temp_cathode_outlet,-0.2414,-0.2041,0.2414,0.2041,-0.0373,0.0373
temp_anode_outlet,-0.0399,-0.0611,0.0399,0.0611,0.0212,0.0212
temp_anode_dewpoint_water,-0.0199,0.0006,0.0199,0.0006,-0.0193,0.0193


In [ ]:
#### Pearson–Spearman Comparison

The coefficients are compared directly to identify predictors whose monotonic association with voltage differs from their linear association.

In [56]:
# ============================================================
# D3.5.3 Visualise Pearson-Spearman Associations with Voltage
# ============================================================

comparison_plot = pearson_spearman_comparison.copy()

# Sort by Pearson coefficient for readable plotting
comparison_plot = comparison_plot.sort_values(
    "pearson_r",
    ascending=True
)

y = np.arange(len(comparison_plot))
bar_height = 0.36

fig, ax = plt.subplots(figsize=(11, 8))

bars_pearson = ax.barh(
    y - bar_height / 2,
    comparison_plot["pearson_r"],
    height=bar_height,
    label="Pearson",
    alpha=0.8
)

bars_spearman = ax.barh(
    y + bar_height / 2,
    comparison_plot["spearman_rho"],
    height=bar_height,
    label="Spearman",
    alpha=0.8
)

# Variable labels
ax.set_yticks(y)
ax.set_yticklabels(comparison_plot.index)

# Zero reference line
ax.axvline(
    0,
    color="black",
    linewidth=1
)

# ------------------------------------------------------------
# Add Pearson value labels outside bars
# ------------------------------------------------------------

for bar, value in zip(
    bars_pearson,
    comparison_plot["pearson_r"]
):
    y_pos = bar.get_y() + bar.get_height() / 2

    if value < 0:
        ax.text(
            value - 0.025,
            y_pos,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=8.5,
            fontweight="bold"
        )
    else:
        ax.text(
            value + 0.025,
            y_pos,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=8.5,
            fontweight="bold"
        )

# ------------------------------------------------------------
# Add Spearman value labels outside bars
# ------------------------------------------------------------

for bar, value in zip(
    bars_spearman,
    comparison_plot["spearman_rho"]
):
    y_pos = bar.get_y() + bar.get_height() / 2

    if value < 0:
        ax.text(
            value - 0.025,
            y_pos,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=8.5,
            fontweight="bold"
        )
    else:
        ax.text(
            value + 0.025,
            y_pos,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=8.5,
            fontweight="bold"
        )

# Extra horizontal space for labels
ax.set_xlim(-1.15, 1.15)

ax.set_xlabel("Association Coefficient")
ax.set_ylabel("Operational Variable")

ax.set_title(
    "Pearson–Spearman Comparison of Associations with Voltage",
    fontsize=13,
    pad=12
)

ax.legend()

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()
plt.show()

<Figure size 1100x800 with 1 Axes>

In [ ]:
#### Difference in Association Strength

Differences in absolute Pearson and Spearman coefficients are examined to identify relationships that may not be adequately represented by a purely linear measure.

In [55]:
# ============================================================
# Visualise Pearson-Spearman Strength Differences
# ============================================================

difference_plot = pearson_spearman_comparison.sort_values(
    "strength_difference",
    ascending=True
)

values = difference_plot["strength_difference"]

colors = [
    "red" if value < 0 else "blue"
    for value in values
]

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(
    difference_plot.index,
    values,
    color=colors,
    alpha=0.8,
    edgecolor="black",
    linewidth=0.5
)

ax.axvline(
    0,
    color="black",
    linewidth=1
)

for bar, value in zip(bars, values):

    y_position = bar.get_y() + bar.get_height() / 2

    if value >= 0:
        ax.text(
            value + 0.005,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=9,
            fontweight="bold"
        )
    else:
        ax.text(
            value - 0.005,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=9,
            fontweight="bold"
        )

ax.set_xlabel("Difference in Association Strength  (|Spearman| − |Pearson|)")
ax.set_ylabel("Operational Variable")

ax.set_title(
    "Difference Between Monotonic and Linear Association Strength"
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()
plt.show()

<Figure size 1000x700 with 1 Axes>

In [ ]:
## D3.6 Mutual Information Analysis

Mutual Information (MI) is used to assess potential non-linear dependencies between the candidate operational variables and stack voltage.

In [65]:
# ============================================================
# D3.6.2 Prepare Data for Mutual Information
# ============================================================

# Candidate operational predictors
candidate_predictors = [
    col for col in association_df.columns
    if col not in [TARGET_VARIABLE, "power", "operating_hour"]
]

# Predictor matrix and target
X_mi = association_df[candidate_predictors].copy()
y_mi = association_df[TARGET_VARIABLE].copy()

print("=" * 60)
print("Mutual Information Analysis Data")
print("=" * 60)

print(f"Target variable: {TARGET_VARIABLE}")
print(f"Number of candidate predictors: {X_mi.shape[1]}")
print(f"Number of observations: {X_mi.shape[0]}")

display(X_mi.head())

Mutual Information Analysis Data
Target variable: voltage
Number of candidate predictors: 14
Number of observations: 3629680


,current,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
0,0.0000,109.9013,110.3273,109.8001,108.7921,83.1282,54.4647,71.9515,39.4532,64.4643,69.6736,56.3375,0.0700,0.2910
1,0.0000,110.1037,110.3273,109.8001,108.8934,83.0788,54.4647,71.9021,39.3870,64.4390,69.6612,56.3249,0.0700,0.2910
2,0.0000,110.3060,110.3273,109.8001,108.7921,83.0788,54.5293,71.8897,39.4135,64.4866,69.6859,56.2997,0.0700,0.2910
3,0.0000,110.1037,110.3273,109.8001,108.7921,83.1035,54.4777,71.9391,39.3870,64.4390,69.6859,56.3249,0.0700,0.2910
4,0.0000,109.9013,110.3273,109.6990,108.8934,83.0911,54.4777,71.8897,39.3738,64.4895,69.6612,56.2492,0.0700,0.2910


In [ ]:
### D3.6.3 Mutual Information Estimation

Mutual Information is estimated using the complete analytical dataset to assess non-linear statistical dependence between each candidate operational predictor and voltage.

In [68]:
# ============================================================
# D3.6.3 Calculate Mutual Information with Voltage
# ============================================================

from sklearn.feature_selection import mutual_info_regression

# ------------------------------------------------------------
# Calculate Mutual Information using all observations
# ------------------------------------------------------------

mi_scores = mutual_info_regression(
    X_mi,
    y_mi,
    random_state=42
)

# ------------------------------------------------------------
# Create ranked Mutual Information table
# ------------------------------------------------------------

mi_results = pd.DataFrame({
    "variable": X_mi.columns,
    "mutual_information": mi_scores
})

# Rank from strongest to weakest dependence
mi_results = (
    mi_results
    .sort_values(
        "mutual_information",
        ascending=False
    )
    .reset_index(drop=True)
)

# Start ranking from 1
mi_results.index = mi_results.index + 1
mi_results.index.name = "rank"

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 60)
print("Mutual Information Associations with Voltage")
print("=" * 60)

print(f"Target variable        : {TARGET_VARIABLE}")
print(f"Number of predictors   : {X_mi.shape[1]}")
print(f"Number of observations : {X_mi.shape[0]:,}")

display(mi_results.round(4))

Mutual Information Associations with Voltage
Target variable        : voltage
Number of predictors   : 14
Number of observations : 3,629,680


,variable,mutual_information
rank,,
1,current,1.5505
2,total_cathode_stack_flow,1.4545
3,total_anode_stack_flow,1.4411
4,temp_anode_endplate,0.9053
5,pressure_cathode_inlet,0.8286
6,temp_cathode_inlet,0.7614
7,pressure_cathode_outlet,0.6866
8,temp_anode_inlet,0.5244
9,temp_anode_outlet,0.3541


In [ ]:
### D3.6.4 Visualise Mutual Information Associations

The Mutual Information scores are visualised to compare the strength of general statistical dependence between each candidate operational predictor and voltage.

In [69]:
# ============================================================
# D3.6.4 Visualise Mutual Information Associations with Voltage
# ============================================================

# Sort for horizontal plotting
mi_plot = mi_results.sort_values(
    "mutual_information",
    ascending=True
).copy()

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(
    mi_plot["variable"],
    mi_plot["mutual_information"],
    alpha=0.85
)

# ------------------------------------------------------------
# Add MI values outside bars
# ------------------------------------------------------------

max_mi = mi_plot["mutual_information"].max()

for bar, value in zip(
    bars,
    mi_plot["mutual_information"]
):
    ax.text(
        value + (max_mi * 0.015),
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold"
    )

# Extra space for value labels
ax.set_xlim(
    0,
    max_mi * 1.15
)

ax.set_xlabel("Mutual Information Score")
ax.set_ylabel("Operational Variable")

ax.set_title(
    "Mutual Information Associations Between Operational Variables and Voltage",
    fontsize=13,
    pad=12
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()
plt.show()

<Figure size 1000x700 with 1 Axes>

In [ ]:
### D3.7 Integrated Association Analysis

Pearson, Spearman, and Mutual Information results are combined to compare linear, monotonic, and general dependence with voltage.

In [70]:
# ============================================================
# D3.7.1 Create Integrated Association Table
# ============================================================

# Pearson results
integrated_association = candidate_voltage_pearson[
    ["pearson_r"]
].copy()

# Add Spearman results
integrated_association = integrated_association.join(
    candidate_voltage_spearman[["spearman_rho"]],
    how="inner"
)

# Prepare MI results using variable as index
mi_for_merge = (
    mi_results
    .set_index("variable")[["mutual_information"]]
)

# Add Mutual Information results
integrated_association = integrated_association.join(
    mi_for_merge,
    how="inner"
)

# Absolute Pearson and Spearman strengths
integrated_association["abs_pearson"] = (
    integrated_association["pearson_r"].abs()
)

integrated_association["abs_spearman"] = (
    integrated_association["spearman_rho"].abs()
)

# Pearson-Spearman strength difference
integrated_association["spearman_minus_pearson_strength"] = (
    integrated_association["abs_spearman"]
    - integrated_association["abs_pearson"]
)

# Rank variables independently under each method
integrated_association["pearson_rank"] = (
    integrated_association["abs_pearson"]
    .rank(method="min", ascending=False)
    .astype(int)
)

integrated_association["spearman_rank"] = (
    integrated_association["abs_spearman"]
    .rank(method="min", ascending=False)
    .astype(int)
)

integrated_association["mi_rank"] = (
    integrated_association["mutual_information"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# Sort by MI rank for initial inspection
integrated_association = integrated_association.sort_values(
    "mi_rank"
)

print("=" * 75)
print("Integrated Association Analysis with Voltage")
print("=" * 75)

display(integrated_association.round(4))

Integrated Association Analysis with Voltage


,pearson_r,spearman_rho,mutual_information,abs_pearson,abs_spearman,spearman_minus_pearson_strength,pearson_rank,spearman_rank,mi_rank
current,-0.9702,-0.9284,1.5505,0.9702,0.9284,-0.0417,1,1,1
total_cathode_stack_flow,-0.9222,-0.9276,1.4545,0.9222,0.9276,0.0054,2,3,2
total_anode_stack_flow,-0.9218,-0.9281,1.4411,0.9218,0.9281,0.0063,3,2,3
temp_anode_endplate,-0.4041,-0.4528,0.9053,0.4041,0.4528,0.0487,6,6,4
pressure_cathode_inlet,-0.6703,-0.7612,0.8286,0.6703,0.7612,0.0909,5,4,5
temp_cathode_inlet,-0.7423,-0.6558,0.7614,0.7423,0.6558,-0.0864,4,5,6
pressure_cathode_outlet,0.3024,0.2239,0.6866,0.3024,0.2239,-0.0786,8,10,7
temp_anode_inlet,-0.3859,-0.3709,0.5244,0.3859,0.3709,-0.0150,7,8,8
temp_anode_outlet,-0.0399,-0.0611,0.3541,0.0399,0.0611,0.0212,12,12,9
temp_cathode_dewpoint_water,-0.0146,-0.0214,0.3400,0.0146,0.0214,0.0067,14,13,10


In [ ]:
### D3.7.2 Cross-Method Rank Comparison

Predictor rankings from Pearson, Spearman, and Mutual Information are compared to assess consistency across association methods.

In [71]:
# ============================================================
# D3.7.2 Visualise Cross-Method Association Rankings
# ============================================================

rank_plot = integrated_association[
    ["pearson_rank", "spearman_rank", "mi_rank"]
].copy()

# Sort using average rank only for visual ordering
rank_plot["average_rank"] = rank_plot[
    ["pearson_rank", "spearman_rank", "mi_rank"]
].mean(axis=1)

rank_plot = rank_plot.sort_values(
    "average_rank",
    ascending=False
)

y = np.arange(len(rank_plot))
bar_height = 0.25

fig, ax = plt.subplots(figsize=(11, 8))

bars_pearson = ax.barh(
    y - bar_height,
    rank_plot["pearson_rank"],
    height=bar_height,
    label="Pearson"
)

bars_spearman = ax.barh(
    y,
    rank_plot["spearman_rank"],
    height=bar_height,
    label="Spearman"
)

bars_mi = ax.barh(
    y + bar_height,
    rank_plot["mi_rank"],
    height=bar_height,
    label="Mutual Information"
)

# Variable labels
ax.set_yticks(y)
ax.set_yticklabels(rank_plot.index)

# Add rank numbers
for bars in [bars_pearson, bars_spearman, bars_mi]:
    for bar in bars:
        value = bar.get_width()

        ax.text(
            value + 0.15,
            bar.get_y() + bar.get_height() / 2,
            f"{int(value)}",
            va="center",
            ha="left",
            fontsize=8,
            fontweight="bold"
        )

ax.set_xlabel("Association Rank (1 = Strongest)")
ax.set_ylabel("Operational Variable")

ax.set_title(
    "Cross-Method Ranking of Associations with Voltage",
    fontsize=13,
    pad=12
)

ax.set_xlim(0, len(rank_plot) + 1)

ax.legend()

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()
plt.show()

<Figure size 1100x800 with 1 Axes>

In [ ]:
### D3.7.3 Assess Cross-Method Ranking Consistency

Variation in predictor rankings across Pearson, Spearman, and Mutual Information is assessed to identify consistent and method-dependent associations.

In [72]:
# ============================================================
# D3.7.3 Assess Cross-Method Ranking Consistency
# ============================================================

association_consistency = integrated_association[
    [
        "pearson_rank",
        "spearman_rank",
        "mi_rank"
    ]
].copy()

# Lowest and highest rank assigned by the three methods
association_consistency["best_rank"] = (
    association_consistency[
        ["pearson_rank", "spearman_rank", "mi_rank"]
    ].min(axis=1)
)

association_consistency["worst_rank"] = (
    association_consistency[
        ["pearson_rank", "spearman_rank", "mi_rank"]
    ].max(axis=1)
)

# Range of rankings across methods
association_consistency["rank_range"] = (
    association_consistency["worst_rank"]
    - association_consistency["best_rank"]
)

# Mean rank for descriptive comparison only
association_consistency["mean_rank"] = (
    association_consistency[
        ["pearson_rank", "spearman_rank", "mi_rank"]
    ].mean(axis=1)
)

# Sort from most consistent to least consistent
association_consistency = association_consistency.sort_values(
    ["rank_range", "mean_rank"],
    ascending=[True, True]
)

print("=" * 70)
print("Cross-Method Consistency of Voltage Association Rankings")
print("=" * 70)

display(
    association_consistency.round(2)
)

Cross-Method Consistency of Voltage Association Rankings


,pearson_rank,spearman_rank,mi_rank,best_rank,worst_rank,rank_range,mean_rank
current,1,1,1,1,1,0,1.0000
total_cathode_stack_flow,2,3,2,2,3,1,2.3300
total_anode_stack_flow,3,2,3,2,3,1,2.6700
pressure_cathode_inlet,5,4,5,4,5,1,4.6700
temp_anode_inlet,7,8,8,7,8,1,7.6700
temp_cathode_inlet,4,5,6,4,6,2,5.0000
temp_anode_endplate,6,6,4,4,6,2,5.3300
temp_cathode_outlet,9,11,11,9,11,2,10.3300
temp_anode_dewpoint_water,13,14,12,12,14,2,13.0000
pressure_cathode_outlet,8,10,7,7,10,3,8.3300


In [ ]:
### D3.3.9 Stage-Wise Pearson Relationship Stability

Calculate predictor–voltage Pearson correlations separately at each durability stage to assess whether relationships remain stable or evolve with aging.

In [80]:
# ============================================================
# D3.3.9 Stage-Wise Pearson Relationship Stability
# ============================================================

stagewise_predictors = [
    "current",
    "total_cathode_stack_flow",
    "pressure_cathode_inlet",
    "temp_cathode_inlet",
    "temp_anode_endplate",
    "pressure_anode_inlet"
]

stagewise_pearson_results = []

# ------------------------------------------------------------
# Calculate Pearson correlation separately for each stage
# ------------------------------------------------------------

for stage in sorted(association_df["operating_hour"].unique()):

    stage_data = association_df[
        association_df["operating_hour"] == stage
    ]

    for predictor in stagewise_predictors:

        r_value = stage_data[
            [predictor, TARGET_VARIABLE]
        ].corr(
            method="pearson"
        ).iloc[0, 1]

        stagewise_pearson_results.append({
            "operating_hour": stage,
            "predictor": predictor,
            "pearson_r": r_value,
            "n_observations": len(stage_data)
        })

# ------------------------------------------------------------
# Convert results to DataFrame
# ------------------------------------------------------------

stagewise_pearson = pd.DataFrame(
    stagewise_pearson_results
)

# ------------------------------------------------------------
# Create stage × predictor table
# ------------------------------------------------------------

stagewise_pearson_table = (
    stagewise_pearson
    .pivot(
        index="operating_hour",
        columns="predictor",
        values="pearson_r"
    )
    .sort_index()
)

# Arrange columns in intended order
stagewise_pearson_table = (
    stagewise_pearson_table[
        stagewise_predictors
    ]
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("=" * 70)
print("Stage-Wise Pearson Associations with Voltage")
print("=" * 70)

print(
    f"Durability stages analysed: "
    f"{stagewise_pearson_table.shape[0]}"
)

print(
    f"Predictors analysed: "
    f"{stagewise_pearson_table.shape[1]}"
)

print("\nPearson correlation with voltage at each durability stage:")

display(
    stagewise_pearson_table.round(4)
)

Stage-Wise Pearson Associations with Voltage
Durability stages analysed: 20
Predictors analysed: 6

Pearson correlation with voltage at each durability stage:


predictor,current,total_cathode_stack_flow,pressure_cathode_inlet,temp_cathode_inlet,temp_anode_endplate,pressure_anode_inlet
operating_hour,,,,,,
50,-0.9769,-0.9359,-0.6780,-0.7394,-0.7306,-0.4844
100,-0.9799,-0.9475,-0.6841,-0.7632,-0.7071,-0.4611
150,-0.9779,-0.9366,-0.6944,-0.7467,-0.7222,-0.4908
200,-0.9700,-0.9158,-0.6817,-0.7423,-0.6983,-0.0500
250,-0.9796,-0.9339,-0.6897,-0.7354,-0.7171,-0.3740
300,-0.9796,-0.9303,-0.6885,-0.7471,-0.6630,-0.3696
350,-0.9805,-0.9334,-0.6727,-0.7573,-0.7162,-0.3094
400,-0.9721,-0.9187,-0.6706,-0.7269,-0.6005,-0.2537
450,-0.9775,-0.9235,-0.6697,-0.7650,-0.7074,-0.3231


In [ ]:
### D3.3.10 Visualisation of Stage-Wise Pearson Stability

Visualise predictor–voltage Pearson correlations across durability stages to assess whether the strength and direction of linear relationships remain stable or evolve as operating hours increase.

In [81]:
# ============================================================
# D3.3.10 Visualise Stage-Wise Pearson Stability
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))

# ------------------------------------------------------------
# Plot one line for each predictor
# ------------------------------------------------------------

for predictor in stagewise_predictors:

    ax.plot(
        stagewise_pearson_table.index,
        stagewise_pearson_table[predictor],
        marker="o",
        linewidth=2,
        markersize=5,
        label=predictor
    )

# ------------------------------------------------------------
# Reference line at zero correlation
# ------------------------------------------------------------

ax.axhline(
    0,
    color="black",
    linewidth=1,
    linestyle="--"
)

# ------------------------------------------------------------
# Axis configuration
# ------------------------------------------------------------

ax.set_xlabel(
    "Durability Stage (Operating Hour)",
    fontsize=11
)

ax.set_ylabel(
    "Pearson Correlation with Voltage (r)",
    fontsize=11
)

ax.set_title(
    "Stage-Wise Stability of Linear Associations with Voltage",
    fontsize=13,
    pad=12
)

# Show every durability stage
ax.set_xticks(
    stagewise_pearson_table.index
)

ax.tick_params(
    axis="x",
    rotation=45
)

# Pearson theoretical range
ax.set_ylim(-1.05, 1.05)

# ------------------------------------------------------------
# Grid and legend
# ------------------------------------------------------------

ax.grid(
    linestyle="--",
    alpha=0.35
)

ax.legend(
    title="Operational Variable",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

<Figure size 1200x700 with 1 Axes>

In [ ]:
### D3.3.11 Quantification of Stage-Wise Pearson Stability

Quantify the average strength and stage-to-stage variability of Pearson associations to distinguish stable from stage-dependent predictor–voltage relationships.

In [82]:
# ============================================================
# D3.3.11 Quantification of Stage-Wise Pearson Stability
# ============================================================

pearson_stability_summary = pd.DataFrame({
    "mean_r": stagewise_pearson_table.mean(),
    "mean_abs_r": stagewise_pearson_table.abs().mean(),
    "sd_r": stagewise_pearson_table.std(),
    "min_r": stagewise_pearson_table.min(),
    "max_r": stagewise_pearson_table.max()
})

# Range of stage-wise Pearson coefficients
pearson_stability_summary["r_range"] = (
    pearson_stability_summary["max_r"]
    - pearson_stability_summary["min_r"]
)

# Sort by smallest range = most stable
pearson_stability_summary = (
    pearson_stability_summary
    .sort_values(
        "r_range",
        ascending=True
    )
)

print("=" * 70)
print("Stage-Wise Pearson Stability Summary")
print("=" * 70)

display(
    pearson_stability_summary.round(4)
)

Stage-Wise Pearson Stability Summary


,mean_r,mean_abs_r,sd_r,min_r,max_r,r_range
predictor,,,,,,
current,-0.9773,0.9773,0.0041,-0.9834,-0.9700,0.0134
temp_cathode_inlet,-0.7516,0.7516,0.0117,-0.7690,-0.7269,0.0420
total_cathode_stack_flow,-0.9277,0.9277,0.0098,-0.9475,-0.9048,0.0428
pressure_cathode_inlet,-0.6757,0.6757,0.0102,-0.6944,-0.6488,0.0457
temp_anode_endplate,-0.7081,0.7081,0.0408,-0.7615,-0.6005,0.1610
pressure_anode_inlet,-0.3197,0.3197,0.1420,-0.4908,-0.0500,0.4408


In [ ]:
### Stage-Wise Spearman Associations with Voltage

Evaluate how the monotonic relationships between selected operational predictors and voltage change across durability stages from 50 h to 1000 h.

In [83]:
# ============================================================
# Stage-Wise Spearman Associations with Voltage
# ============================================================

# Use the same predictors selected for the Pearson
# stage-wise stability analysis
stagewise_spearman_results = []

for operating_hour, stage_df in association_df.groupby("operating_hour"):

    for predictor in stagewise_predictors:

        spearman_rho = stage_df[
            [predictor, TARGET_VARIABLE]
        ].corr(method="spearman").iloc[0, 1]

        stagewise_spearman_results.append({
            "operating_hour": operating_hour,
            "predictor": predictor,
            "spearman_rho": spearman_rho
        })

# Convert results to DataFrame
stagewise_spearman_df = pd.DataFrame(
    stagewise_spearman_results
)

# Pivot for stage-wise comparison
stagewise_spearman_table = (
    stagewise_spearman_df
    .pivot(
        index="operating_hour",
        columns="predictor",
        values="spearman_rho"
    )
    .sort_index()
)

# Keep predictor order consistent with Pearson analysis
stagewise_spearman_table = (
    stagewise_spearman_table[stagewise_predictors]
)

print("=" * 70)
print("Stage-Wise Spearman Associations with Voltage")
print("=" * 70)

print(
    "\nSpearman correlation with voltage "
    "at each durability stage:\n"
)

display(
    stagewise_spearman_table.round(4)
)

Stage-Wise Spearman Associations with Voltage

Spearman correlation with voltage at each durability stage:



predictor,current,total_cathode_stack_flow,pressure_cathode_inlet,temp_cathode_inlet,temp_anode_endplate,pressure_anode_inlet
operating_hour,,,,,,
50,-0.9346,-0.9231,-0.7680,-0.6566,-0.6210,-0.4804
100,-0.9375,-0.9334,-0.7881,-0.6612,-0.5436,-0.4927
150,-0.9394,-0.9178,-0.7948,-0.6569,-0.6190,-0.4946
200,-0.9339,-0.9215,-0.7799,-0.6514,-0.5760,-0.4305
250,-0.9307,-0.9305,-0.7943,-0.6286,-0.5817,-0.3945
300,-0.9298,-0.9201,-0.7825,-0.6292,-0.5435,-0.4077
350,-0.9309,-0.9202,-0.7726,-0.6631,-0.5644,-0.3798
400,-0.9268,-0.9177,-0.7604,-0.6047,-0.4348,-0.4145
450,-0.9354,-0.9174,-0.7617,-0.6595,-0.6016,-0.3755


In [84]:
# ============================================================
# Visualise Stage-Wise Spearman Stability
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))

for predictor in stagewise_predictors:

    ax.plot(
        stagewise_spearman_table.index,
        stagewise_spearman_table[predictor],
        marker="o",
        linewidth=2,
        markersize=5,
        label=predictor
    )

# Zero-reference line
ax.axhline(
    0,
    color="black",
    linewidth=1,
    linestyle="--"
)

ax.set_xlabel(
    "Durability Stage (Operating Hour)",
    fontsize=11
)

ax.set_ylabel(
    "Spearman Correlation with Voltage (ρ)",
    fontsize=11
)

ax.set_title(
    "Stage-Wise Stability of Monotonic Associations with Voltage",
    fontsize=13,
    pad=12
)

# Show all durability stages
ax.set_xticks(
    stagewise_spearman_table.index
)

ax.tick_params(
    axis="x",
    rotation=45
)

# Spearman theoretical range
ax.set_ylim(-1.05, 1.05)

ax.grid(
    linestyle="--",
    alpha=0.35
)

ax.legend(
    title="Operational Variable",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

<Figure size 1200x700 with 1 Axes>

In [85]:
# ============================================================
# Quantification of Stage-Wise Spearman Stability
# ============================================================

spearman_stability_summary = pd.DataFrame({
    "mean_rho": stagewise_spearman_table.mean(),
    "mean_abs_rho": stagewise_spearman_table.abs().mean(),
    "sd_rho": stagewise_spearman_table.std(),
    "min_rho": stagewise_spearman_table.min(),
    "max_rho": stagewise_spearman_table.max()
})

# Range of stage-wise Spearman coefficients
spearman_stability_summary["rho_range"] = (
    spearman_stability_summary["max_rho"]
    - spearman_stability_summary["min_rho"]
)

# Sort by smallest range = most stable
spearman_stability_summary = (
    spearman_stability_summary
    .sort_values(
        "rho_range",
        ascending=True
    )
)

print("=" * 70)
print("Stage-Wise Spearman Stability Summary")
print("=" * 70)

display(
    spearman_stability_summary.round(4)
)

Stage-Wise Spearman Stability Summary


,mean_rho,mean_abs_rho,sd_rho,min_rho,max_rho,rho_range
predictor,,,,,,
current,-0.9362,0.9362,0.0071,-0.9505,-0.9188,0.0317
temp_cathode_inlet,-0.6502,0.6502,0.0153,-0.6720,-0.6047,0.0673
pressure_cathode_inlet,-0.7713,0.7713,0.0182,-0.7948,-0.7191,0.0757
total_cathode_stack_flow,-0.9175,0.9175,0.0175,-0.9334,-0.8495,0.0839
pressure_anode_inlet,-0.4023,0.4023,0.0655,-0.4946,-0.2682,0.2264
temp_anode_endplate,-0.5882,0.5882,0.0585,-0.6866,-0.4348,0.2518


In [88]:
# ============================================================
# D3.5.X Pearson-Spearman Stage-Wise Stability Comparison
# ============================================================

# ------------------------------------------------------------
# Select relevant Pearson stability metrics
# ------------------------------------------------------------

pearson_stability = pearson_stability_summary[
    [
        "mean_abs_r",
        "sd_r",
        "r_range"
    ]
].copy()

pearson_stability = pearson_stability.rename(
    columns={
        "mean_abs_r": "pearson_mean_strength",
        "sd_r": "pearson_sd",
        "r_range": "pearson_range"
    }
)

# ------------------------------------------------------------
# Select relevant Spearman stability metrics
# ------------------------------------------------------------

spearman_stability = spearman_stability_summary[
    [
        "mean_abs_rho",
        "sd_rho",
        "rho_range"
    ]
].copy()

spearman_stability = spearman_stability.rename(
    columns={
        "mean_abs_rho": "spearman_mean_strength",
        "sd_rho": "spearman_sd",
        "rho_range": "spearman_range"
    }
)

# ------------------------------------------------------------
# Combine Pearson and Spearman stability summaries
# ------------------------------------------------------------

stage_stability_comparison = pearson_stability.join(
    spearman_stability,
    how="inner"
)

# ------------------------------------------------------------
# Compare average association strength
# Spearman |rho| - Pearson |r|
# ------------------------------------------------------------

stage_stability_comparison["strength_difference"] = (
    stage_stability_comparison["spearman_mean_strength"]
    - stage_stability_comparison["pearson_mean_strength"]
)

# ------------------------------------------------------------
# Compare stage-wise variability
# Spearman SD - Pearson SD
# ------------------------------------------------------------

stage_stability_comparison["sd_difference"] = (
    stage_stability_comparison["spearman_sd"]
    - stage_stability_comparison["pearson_sd"]
)

# ------------------------------------------------------------
# Compare total stage-wise range
# Spearman range - Pearson range
# ------------------------------------------------------------

stage_stability_comparison["range_difference"] = (
    stage_stability_comparison["spearman_range"]
    - stage_stability_comparison["pearson_range"]
)

# ------------------------------------------------------------
# Sort by strongest mean Pearson association
# ------------------------------------------------------------

stage_stability_comparison = (
    stage_stability_comparison
    .sort_values(
        "pearson_mean_strength",
        ascending=False
    )
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 75)
print("Pearson-Spearman Stage-Wise Stability Comparison")
print("=" * 75)

display(
    stage_stability_comparison.round(4)
)

Pearson-Spearman Stage-Wise Stability Comparison


,pearson_mean_strength,pearson_sd,pearson_range,spearman_mean_strength,spearman_sd,spearman_range,strength_difference,sd_difference,range_difference
predictor,,,,,,,,,
current,0.9773,0.0041,0.0134,0.9362,0.0071,0.0317,-0.0411,0.0030,0.0183
total_cathode_stack_flow,0.9277,0.0098,0.0428,0.9175,0.0175,0.0839,-0.0102,0.0077,0.0411
temp_cathode_inlet,0.7516,0.0117,0.0420,0.6502,0.0153,0.0673,-0.1014,0.0036,0.0253
temp_anode_endplate,0.7081,0.0408,0.1610,0.5882,0.0585,0.2518,-0.1199,0.0177,0.0908
pressure_cathode_inlet,0.6757,0.0102,0.0457,0.7713,0.0182,0.0757,0.0955,0.0080,0.0300
pressure_anode_inlet,0.3197,0.1420,0.4408,0.4023,0.0655,0.2264,0.0826,-0.0765,-0.2144


In [ ]:
### D3.X.2 Identification of Strong Predictor-to-Predictor Associations

This step examines relationships **between candidate predictors**, rather than their relationships with voltage.

The purpose is to identify variables that are strongly coupled and may therefore contain overlapping or redundant information. Pearson and Spearman associations are extracted for every unique predictor pair and ranked by association strength.

High association is treated as a **flag for further investigation**, not as an automatic criterion for removing a variable. Statistical coupling will later be interpreted together with PEMFC physical relationships and voltage relevance.

In [90]:
# ============================================================
# D3.X.2 Identify Strong Predictor-to-Predictor Associations
# ============================================================

# Candidate predictors only:
# exclude target voltage and mathematically dependent power
redundancy_predictors = [
    col for col in association_df.columns
    if col not in [TARGET_VARIABLE, "power", STAGE_VARIABLE]
]

# ------------------------------------------------------------
# Extract every unique predictor pair
# ------------------------------------------------------------

predictor_pair_results = []

for i in range(len(redundancy_predictors)):
    for j in range(i + 1, len(redundancy_predictors)):

        variable_1 = redundancy_predictors[i]
        variable_2 = redundancy_predictors[j]

        pearson_value = pearson_matrix.loc[
            variable_1, variable_2
        ]

        spearman_value = spearman_corr.loc[
            variable_1, variable_2
        ]

        predictor_pair_results.append({
            "variable_1": variable_1,
            "variable_2": variable_2,
            "pearson_r": pearson_value,
            "spearman_rho": spearman_value,
            "abs_pearson": abs(pearson_value),
            "abs_spearman": abs(spearman_value)
        })

# ------------------------------------------------------------
# Create pairwise association table
# ------------------------------------------------------------

predictor_pair_associations = pd.DataFrame(
    predictor_pair_results
)

# Combined descriptive strength measure
predictor_pair_associations["mean_abs_association"] = (
    predictor_pair_associations[
        ["abs_pearson", "abs_spearman"]
    ].mean(axis=1)
)

# Difference between monotonic and linear association strength
predictor_pair_associations["spearman_minus_pearson_strength"] = (
    predictor_pair_associations["abs_spearman"]
    - predictor_pair_associations["abs_pearson"]
)

# Rank strongest coupled pairs first
predictor_pair_associations = (
    predictor_pair_associations
    .sort_values(
        "mean_abs_association",
        ascending=False
    )
    .reset_index(drop=True)
)

predictor_pair_associations.index += 1
predictor_pair_associations.index.name = "rank"

# ------------------------------------------------------------
# Display strongest relationships
# ------------------------------------------------------------

print("=" * 75)
print("Strongest Predictor-to-Predictor Associations")
print("=" * 75)

print(f"Candidate predictors analysed : {len(redundancy_predictors)}")
print(
    f"Unique predictor pairs       : "
    f"{len(predictor_pair_associations):,}"
)

display(
    predictor_pair_associations
    .head(20)
    .round(4)
)

Strongest Predictor-to-Predictor Associations
Candidate predictors analysed : 14
Unique predictor pairs       : 91


,variable_1,variable_2,pearson_r,spearman_rho,abs_pearson,abs_spearman,mean_abs_association,spearman_minus_pearson_strength
rank,,,,,,,,
1,total_anode_stack_flow,total_cathode_stack_flow,1.0000,0.9974,1.0000,0.9974,0.9987,-0.0026
2,current,total_cathode_stack_flow,0.9805,0.9167,0.9805,0.9167,0.9486,-0.0637
3,current,total_anode_stack_flow,0.9803,0.9165,0.9803,0.9165,0.9484,-0.0637
4,temp_cathode_inlet,total_cathode_stack_flow,0.8099,0.7577,0.8099,0.7577,0.7838,-0.0522
5,temp_cathode_inlet,total_anode_stack_flow,0.8096,0.7553,0.8096,0.7553,0.7824,-0.0543
6,current,temp_cathode_inlet,0.7999,0.6942,0.7999,0.6942,0.7471,-0.1057
7,current,pressure_cathode_inlet,0.6092,0.7619,0.6092,0.7619,0.6856,0.1527
8,pressure_cathode_inlet,total_cathode_stack_flow,0.5691,0.7814,0.5691,0.7814,0.6753,0.2123
9,pressure_cathode_inlet,total_anode_stack_flow,0.5688,0.7806,0.5688,0.7806,0.6747,0.2119


In [ ]:
### D3.X.3 Candidate Redundancy Groups

Group strongly associated predictors to identify sets of variables that may represent overlapping operational information.

In [91]:
# ============================================================
# D3.X.3 Identify Candidate Redundancy Groups
# ============================================================

# Use a descriptive screening threshold only to flag
# strongly coupled predictor pairs for further investigation.
redundancy_threshold = 0.80

candidate_redundant_pairs = (
    predictor_pair_associations[
        predictor_pair_associations["mean_abs_association"]
        >= redundancy_threshold
    ]
    .copy()
)

print("=" * 75)
print("Candidate Strongly Coupled Predictor Pairs")
print("=" * 75)

print(
    f"Descriptive screening threshold : "
    f"{redundancy_threshold:.2f}"
)

print(
    f"Candidate pairs identified      : "
    f"{len(candidate_redundant_pairs)}"
)

display(
    candidate_redundant_pairs.round(4)
)

Candidate Strongly Coupled Predictor Pairs
Descriptive screening threshold : 0.80
Candidate pairs identified      : 3


,variable_1,variable_2,pearson_r,spearman_rho,abs_pearson,abs_spearman,mean_abs_association,spearman_minus_pearson_strength
rank,,,,,,,,
1,total_anode_stack_flow,total_cathode_stack_flow,1.0000,0.9974,1.0000,0.9974,0.9987,-0.0026
2,current,total_cathode_stack_flow,0.9805,0.9167,0.9805,0.9167,0.9486,-0.0637
3,current,total_anode_stack_flow,0.9803,0.9165,0.9803,0.9165,0.9484,-0.0637


In [ ]:
### D3.X.4 Identification of Candidate Redundancy Groups

The strongly coupled predictor pairs are now combined into connected groups. This identifies sets of operational variables that may contain overlapping information rather than considering each pair independently.

In [92]:
# ============================================================
# D3.X.4 Identify Candidate Redundancy Groups
# ============================================================

# Build an undirected graph from the strongly coupled pairs
redundancy_graph = {}

for _, row in candidate_redundant_pairs.iterrows():

    var1 = row["variable_1"]
    var2 = row["variable_2"]

    redundancy_graph.setdefault(var1, set()).add(var2)
    redundancy_graph.setdefault(var2, set()).add(var1)


# ------------------------------------------------------------
# Find connected groups
# ------------------------------------------------------------

visited = set()
redundancy_groups = []

for variable in redundancy_graph:

    if variable in visited:
        continue

    group = set()
    stack = [variable]

    while stack:

        current_variable = stack.pop()

        if current_variable in visited:
            continue

        visited.add(current_variable)
        group.add(current_variable)

        stack.extend(
            redundancy_graph.get(current_variable, set()) - visited
        )

    if len(group) > 1:
        redundancy_groups.append(sorted(group))


# ------------------------------------------------------------
# Create summary table
# ------------------------------------------------------------

redundancy_group_summary = pd.DataFrame({
    "group": range(1, len(redundancy_groups) + 1),
    "variables": [", ".join(group) for group in redundancy_groups],
    "number_of_variables": [len(group) for group in redundancy_groups]
})

print("=" * 75)
print("Candidate Redundancy Groups")
print("=" * 75)

print(f"Number of candidate groups identified: {len(redundancy_groups)}")

display(redundancy_group_summary)

Candidate Redundancy Groups
Number of candidate groups identified: 1


,group,variables,number_of_variables
0,1,"current, total_anode_stack_flow, total_cathode...",3


In [ ]:
### D3.X.5 Within-Group Association Assessment

Quantify the pairwise association strengths within each candidate redundancy group. This provides a compact assessment of how strongly the variables within each identified group overlap in their observed behaviour.

In [93]:
# ============================================================
# D3.X.5 Quantify Within-Group Associations
# ============================================================

within_group_results = []

for group_id, group_variables in enumerate(
    redundancy_groups,
    start=1
):

    # All unique pairs within the group
    for i in range(len(group_variables)):
        for j in range(i + 1, len(group_variables)):

            var1 = group_variables[i]
            var2 = group_variables[j]

            # Find the corresponding pair in the existing
            # predictor-to-predictor association table
            pair_row = predictor_pair_associations[
                (
                    (predictor_pair_associations["variable_1"] == var1) &
                    (predictor_pair_associations["variable_2"] == var2)
                )
                |
                (
                    (predictor_pair_associations["variable_1"] == var2) &
                    (predictor_pair_associations["variable_2"] == var1)
                )
            ]

            if not pair_row.empty:

                row = pair_row.iloc[0]

                within_group_results.append({
                    "group": group_id,
                    "variable_1": var1,
                    "variable_2": var2,
                    "pearson_r": row["pearson_r"],
                    "spearman_rho": row["spearman_rho"],
                    "mean_abs_association": row["mean_abs_association"]
                })


# Convert to DataFrame
within_group_associations = pd.DataFrame(
    within_group_results
)


# ------------------------------------------------------------
# Group-level summary
# ------------------------------------------------------------

redundancy_strength_summary = (
    within_group_associations
    .groupby("group")
    .agg(
        number_of_pairs=("mean_abs_association", "size"),
        mean_within_group_association=("mean_abs_association", "mean"),
        minimum_within_group_association=("mean_abs_association", "min"),
        maximum_within_group_association=("mean_abs_association", "max")
    )
)


print("=" * 75)
print("Within-Group Predictor Associations")
print("=" * 75)

display(
    within_group_associations.round(4)
)

print("\n" + "=" * 75)
print("Candidate Redundancy Group Strength Summary")
print("=" * 75)

display(
    redundancy_strength_summary.round(4)
)

Within-Group Predictor Associations


,group,variable_1,variable_2,pearson_r,spearman_rho,mean_abs_association
0,1,current,total_anode_stack_flow,0.9803,0.9165,0.9484
1,1,current,total_cathode_stack_flow,0.9805,0.9167,0.9486
2,1,total_anode_stack_flow,total_cathode_stack_flow,1.0000,0.9974,0.9987



Candidate Redundancy Group Strength Summary


,number_of_pairs,mean_within_group_association,minimum_within_group_association,maximum_within_group_association
group,,,,
1,3,0.9652,0.9484,0.9987


In [ ]:
### D3.X.6 Identification of Special Association Structures

Predictor pairs with substantial disagreement between Pearson and Spearman association strengths are identified for further investigation. These relationships may contain non-linear, regime-dependent, clustered, or otherwise unusual association structures and should not be classified as simple redundancy based on one correlation measure alone.

In [94]:
# ============================================================
# D3.X.6 Identify Special / Discordant Predictor Pairs
# ============================================================

# Absolute disagreement between Pearson and Spearman strength
predictor_pair_associations["method_disagreement"] = (
    predictor_pair_associations["abs_spearman"]
    - predictor_pair_associations["abs_pearson"]
).abs()


# ------------------------------------------------------------
# Rank pairs by disagreement
# ------------------------------------------------------------

special_pair_candidates = (
    predictor_pair_associations[
        [
            "variable_1",
            "variable_2",
            "pearson_r",
            "spearman_rho",
            "abs_pearson",
            "abs_spearman",
            "method_disagreement"
        ]
    ]
    .sort_values(
        "method_disagreement",
        ascending=False
    )
    .reset_index(drop=True)
)

special_pair_candidates.index = (
    special_pair_candidates.index + 1
)

special_pair_candidates.index.name = "rank"


print("=" * 75)
print("Predictor Pairs with Largest Pearson-Spearman Disagreement")
print("=" * 75)

display(
    special_pair_candidates.head(15).round(4)
)

Predictor Pairs with Largest Pearson-Spearman Disagreement


,variable_1,variable_2,pearson_r,spearman_rho,abs_pearson,abs_spearman,method_disagreement
rank,,,,,,,
1,pressure_anode_inlet,pressure_anode_outlet,0.9441,0.2599,0.9441,0.2599,0.6842
2,pressure_anode_inlet,pressure_cathode_inlet,0.1683,0.5161,0.1683,0.5161,0.3478
3,pressure_anode_outlet,temp_anode_outlet,-0.0906,-0.4309,0.0906,0.4309,0.3404
4,pressure_anode_outlet,pressure_cathode_outlet,0.1905,0.4737,0.1905,0.4737,0.2832
5,pressure_anode_inlet,total_anode_stack_flow,0.1048,0.3875,0.1048,0.3875,0.2827
6,pressure_anode_inlet,total_cathode_stack_flow,0.1048,0.3868,0.1048,0.3868,0.2820
7,current,pressure_anode_inlet,0.1083,0.3730,0.1083,0.3730,0.2647
8,pressure_anode_outlet,temp_cathode_inlet,-0.1280,-0.3858,0.1280,0.3858,0.2578
9,pressure_cathode_outlet,total_anode_stack_flow,-0.5256,-0.3010,0.5256,0.3010,0.2246


In [ ]:
### D3.3.X Visual Assessment of Special Predictor Relationships

Predictor pairs showing large disagreement between Pearson and Spearman association strengths are examined visually.

The purpose is to determine whether the disagreement may arise from relationship structure such as non-linearity, clustering, restricted operating ranges, discrete operating states, or other regime-dependent behaviour.

A stage-stratified sample is used for visualisation so that all durability stages are represented while avoiding overplotting from the full dataset. The statistical association coefficients remain those calculated from the full dataset.

In [95]:
# ============================================================
# D3.3.X Visual Assessment of Special Predictor Relationships
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Select predictor pairs with largest Pearson-Spearman
# disagreement
# ------------------------------------------------------------

top_n_special_pairs = 6

special_pairs = (
    predictor_pair_associations
    .sort_values("method_disagreement", ascending=False)
    .head(top_n_special_pairs)
    .copy()
)

print("=" * 70)
print("Special Predictor Pairs Selected for Visual Assessment")
print("=" * 70)

display(
    special_pairs[
        [
            "variable_1",
            "variable_2",
            "pearson_r",
            "spearman_rho",
            "method_disagreement"
        ]
    ].round(4)
)


# ------------------------------------------------------------
# Create stage-stratified sample
# ------------------------------------------------------------

samples_per_stage_special = 1500

special_visual_sample = (
    association_df
    .groupby("operating_hour", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(samples_per_stage_special, len(x)),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print(
    f"\nObservations used for visualisation: "
    f"{len(special_visual_sample):,}"
)


# ------------------------------------------------------------
# Plot selected relationships
# ------------------------------------------------------------

fig, axes = plt.subplots(
    3,
    2,
    figsize=(15, 16)
)

axes = axes.flatten()

for ax, (_, row) in zip(axes, special_pairs.iterrows()):

    x_var = row["variable_1"]
    y_var = row["variable_2"]

    x = special_visual_sample[x_var]
    y = special_visual_sample[y_var]

    ax.scatter(
        x,
        y,
        alpha=0.15,
        s=8
    )

    # Linear reference line
    valid = x.notna() & y.notna()

    if valid.sum() > 1:

        coefficients = np.polyfit(
            x[valid],
            y[valid],
            1
        )

        x_line = np.linspace(
            x[valid].min(),
            x[valid].max(),
            100
        )

        y_line = (
            coefficients[0] * x_line
            + coefficients[1]
        )

        ax.plot(
            x_line,
            y_line,
            linewidth=2
        )

    ax.set_xlabel(x_var)
    ax.set_ylabel(y_var)

    ax.set_title(
        f"{x_var} vs {y_var}\n"
        f"Pearson = {row['pearson_r']:.3f}, "
        f"Spearman = {row['spearman_rho']:.3f}"
    )

    ax.grid(
        alpha=0.25,
        linestyle="--"
    )

plt.suptitle(
    "Visual Assessment of Predictor Pairs with Largest "
    "Pearson–Spearman Disagreement",
    fontsize=16
)

plt.tight_layout()

plt.show()

Special Predictor Pairs Selected for Visual Assessment


,variable_1,variable_2,pearson_r,spearman_rho,method_disagreement
rank,,,,,
11,pressure_anode_inlet,pressure_anode_outlet,0.9441,0.2599,0.6842
29,pressure_anode_inlet,pressure_cathode_inlet,0.1683,0.5161,0.3478
37,pressure_anode_outlet,temp_anode_outlet,-0.0906,-0.4309,0.3404
30,pressure_anode_outlet,pressure_cathode_outlet,0.1905,0.4737,0.2832
40,pressure_anode_inlet,total_anode_stack_flow,0.1048,0.3875,0.2827
41,pressure_anode_inlet,total_cathode_stack_flow,0.1048,0.3868,0.2820



Observations used for visualisation: 30,000


<Figure size 1500x1600 with 6 Axes>

In [ ]:
### D3.X.7 Variable-Level Redundancy and Special-Relationship Summary

The pairwise redundancy and Pearson–Spearman disagreement results are consolidated at the individual-variable level.

This step identifies which predictors belong to candidate redundancy groups and which predictors repeatedly participate in unusual association structures. The resulting table provides a structured screening summary for later feature-selection and feature-engineering decisions; no variables are removed at this stage.

In [97]:
# ============================================================
# D3.X.7 Variable-Level Redundancy and Special-Relationship Summary
# ============================================================

# Use the same candidate predictors defined earlier
predictors = redundancy_predictors.copy()

# Create summary table
variable_screening_summary = pd.DataFrame(
    index=predictors
)

variable_screening_summary.index.name = "variable"


# ------------------------------------------------------------
# 1. Candidate redundancy-group membership
# ------------------------------------------------------------

redundancy_members = set()

# redundancy_groups is a LIST of variable groups
for group_variables in redundancy_groups:

    redundancy_members.update(group_variables)


variable_screening_summary["in_redundancy_group"] = (
    variable_screening_summary.index
    .isin(redundancy_members)
)


# ------------------------------------------------------------
# 2. Count strongly coupled relationships
# ------------------------------------------------------------

strong_pair_counts = {
    variable: 0
    for variable in predictors
}

for _, row in candidate_redundant_pairs.iterrows():

    var1 = row["variable_1"]
    var2 = row["variable_2"]

    if var1 in strong_pair_counts:
        strong_pair_counts[var1] += 1

    if var2 in strong_pair_counts:
        strong_pair_counts[var2] += 1


variable_screening_summary["strong_pair_count"] = (
    variable_screening_summary.index
    .map(strong_pair_counts)
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 3. Count special / discordant relationships
# ------------------------------------------------------------

# Use the same six pairs that were visually assessed
visually_assessed_special_pairs = (
    special_pair_candidates
    .head(6)
    .copy()
)

special_pair_counts = {
    variable: 0
    for variable in predictors
}

for _, row in visually_assessed_special_pairs.iterrows():

    var1 = row["variable_1"]
    var2 = row["variable_2"]

    if var1 in special_pair_counts:
        special_pair_counts[var1] += 1

    if var2 in special_pair_counts:
        special_pair_counts[var2] += 1


variable_screening_summary["special_pair_count"] = (
    variable_screening_summary.index
    .map(special_pair_counts)
    .fillna(0)
    .astype(int)
)

variable_screening_summary["in_special_relationship"] = (
    variable_screening_summary["special_pair_count"] > 0
)


# ------------------------------------------------------------
# 4. Overall screening category
# ------------------------------------------------------------

def classify_screening_status(row):

    if (
        row["in_redundancy_group"]
        and row["in_special_relationship"]
    ):
        return "Redundancy + special structure"

    elif row["in_redundancy_group"]:
        return "Candidate redundancy"

    elif row["in_special_relationship"]:
        return "Special relationship structure"

    else:
        return "No major flag"


variable_screening_summary["screening_status"] = (
    variable_screening_summary.apply(
        classify_screening_status,
        axis=1
    )
)


# ------------------------------------------------------------
# 5. Sort for readable output
# ------------------------------------------------------------

variable_screening_summary = (
    variable_screening_summary
    .sort_values(
        [
            "in_redundancy_group",
            "special_pair_count",
            "strong_pair_count"
        ],
        ascending=[False, False, False]
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 75)
print("Variable-Level Redundancy and Special-Relationship Summary")
print("=" * 75)

print(
    f"Candidate predictors assessed : "
    f"{len(variable_screening_summary)}"
)

print(
    f"Predictors in redundancy groups : "
    f"{variable_screening_summary['in_redundancy_group'].sum()}"
)

print(
    f"Predictors in special relationships : "
    f"{variable_screening_summary['in_special_relationship'].sum()}"
)

display(variable_screening_summary)

Variable-Level Redundancy and Special-Relationship Summary
Candidate predictors assessed : 14
Predictors in redundancy groups : 3
Predictors in special relationships : 7


,in_redundancy_group,strong_pair_count,special_pair_count,in_special_relationship,screening_status
variable,,,,,
total_anode_stack_flow,True,2,1,True,Redundancy + special structure
total_cathode_stack_flow,True,2,1,True,Redundancy + special structure
current,True,2,0,False,Candidate redundancy
pressure_anode_inlet,False,0,4,True,Special relationship structure
pressure_anode_outlet,False,0,3,True,Special relationship structure
pressure_cathode_inlet,False,0,1,True,Special relationship structure
pressure_cathode_outlet,False,0,1,True,Special relationship structure
temp_anode_outlet,False,0,1,True,Special relationship structure
temp_anode_endplate,False,0,0,False,No major flag


In [ ]:
### D3.X.8 Redundancy Flags and Voltage Relevance

Combine redundancy and special-relationship screening with each predictor's association with voltage.

This helps distinguish variables that are strongly related to voltage but potentially redundant from variables that may provide more distinct information.

In [98]:
# ============================================================
# D3.X.8 Combine Redundancy Flags with Voltage Association
# ============================================================

# ------------------------------------------------------------
# Start from variable-level screening summary
# ------------------------------------------------------------

redundancy_voltage_summary = (
    variable_screening_summary
    .copy()
)

# ------------------------------------------------------------
# Add global Pearson association with voltage
# ------------------------------------------------------------

redundancy_voltage_summary["pearson_with_voltage"] = (
    candidate_voltage_pearson[
        "pearson_r"
    ]
)

redundancy_voltage_summary["abs_pearson_with_voltage"] = (
    redundancy_voltage_summary[
        "pearson_with_voltage"
    ].abs()
)

# ------------------------------------------------------------
# Add global Spearman association with voltage
# ------------------------------------------------------------

redundancy_voltage_summary["spearman_with_voltage"] = (
    candidate_voltage_spearman[
        "spearman_rho"
    ]
)

redundancy_voltage_summary["abs_spearman_with_voltage"] = (
    redundancy_voltage_summary[
        "spearman_with_voltage"
    ].abs()
)

# ------------------------------------------------------------
# Add Mutual Information with voltage
# ------------------------------------------------------------

redundancy_voltage_summary["mutual_information"] = (
    integrated_association[
        "mutual_information"
    ]
)

# ------------------------------------------------------------
# Mean Pearson-Spearman voltage-association strength
# Descriptive only
# ------------------------------------------------------------

redundancy_voltage_summary[
    "mean_abs_voltage_association"
] = (
    redundancy_voltage_summary[
        [
            "abs_pearson_with_voltage",
            "abs_spearman_with_voltage"
        ]
    ]
    .mean(axis=1)
)

# ------------------------------------------------------------
# Sort by strongest overall voltage association
# ------------------------------------------------------------

redundancy_voltage_summary = (
    redundancy_voltage_summary
    .sort_values(
        "mean_abs_voltage_association",
        ascending=False
    )
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("Redundancy / Special-Relationship Screening with Voltage Relevance")
print("=" * 80)

display(
    redundancy_voltage_summary[
        [
            "in_redundancy_group",
            "strong_pair_count",
            "in_special_relationship",
            "special_pair_count",
            "pearson_with_voltage",
            "spearman_with_voltage",
            "mutual_information",
            "mean_abs_voltage_association",
            "screening_status"
        ]
    ].round(4)
)

Redundancy / Special-Relationship Screening with Voltage Relevance


,in_redundancy_group,strong_pair_count,in_special_relationship,special_pair_count,pearson_with_voltage,spearman_with_voltage,mutual_information,mean_abs_voltage_association,screening_status
variable,,,,,,,,,
current,True,2,False,0,-0.9702,-0.9284,1.5505,0.9493,Candidate redundancy
total_anode_stack_flow,True,2,True,1,-0.9218,-0.9281,1.4411,0.9249,Redundancy + special structure
total_cathode_stack_flow,True,2,True,1,-0.9222,-0.9276,1.4545,0.9249,Redundancy + special structure
pressure_cathode_inlet,False,0,True,1,-0.6703,-0.7612,0.8286,0.7158,Special relationship structure
temp_cathode_inlet,False,0,False,0,-0.7423,-0.6558,0.7614,0.6990,No major flag
temp_anode_endplate,False,0,False,0,-0.4041,-0.4528,0.9053,0.4284,No major flag
temp_anode_inlet,False,0,False,0,-0.3859,-0.3709,0.5244,0.3784,No major flag
pressure_cathode_outlet,False,0,True,1,0.3024,0.2239,0.6866,0.2631,Special relationship structure
pressure_anode_inlet,False,0,True,4,-0.1128,-0.3904,0.2262,0.2516,Special relationship structure


In [ ]:
### D3.X.9 Predictor Screening Evidence Summary

Consolidate the association, redundancy, and special-relationship evidence into a compact predictor-level summary.

The purpose is not to remove variables at this stage, but to identify which predictors require particular attention during later feature engineering and feature-selection decisions.

In [99]:
# ============================================================
# D3.X.9 Predictor Screening Evidence Summary
# ============================================================

predictor_screening_evidence = (
    redundancy_voltage_summary
    .copy()
)

# ------------------------------------------------------------
# Assign evidence-based structural category
# ------------------------------------------------------------

def classify_predictor_structure(row):

    if (
        row["in_redundancy_group"]
        and row["in_special_relationship"]
    ):
        return "Redundancy + special structure"

    elif row["in_redundancy_group"]:
        return "Candidate redundancy"

    elif row["in_special_relationship"]:
        return "Special relationship structure"

    else:
        return "No structural flag"


predictor_screening_evidence["structural_category"] = (
    predictor_screening_evidence.apply(
        classify_predictor_structure,
        axis=1
    )
)

# ------------------------------------------------------------
# Add association-method agreement with voltage
# ------------------------------------------------------------

predictor_screening_evidence[
    "pearson_spearman_strength_difference"
] = (
    predictor_screening_evidence[
        "abs_spearman_with_voltage"
    ]
    -
    predictor_screening_evidence[
        "abs_pearson_with_voltage"
    ]
)

# ------------------------------------------------------------
# Rank predictors by voltage relevance
# using the existing integrated association ranking
# ------------------------------------------------------------

predictor_screening_evidence["integrated_mean_rank"] = (
    association_summary["mean_rank"]
)

# ------------------------------------------------------------
# Select final screening columns
# ------------------------------------------------------------

predictor_screening_evidence = (
    predictor_screening_evidence[
        [
            "pearson_with_voltage",
            "spearman_with_voltage",
            "mutual_information",
            "mean_abs_voltage_association",
            "integrated_mean_rank",
            "in_redundancy_group",
            "strong_pair_count",
            "in_special_relationship",
            "special_pair_count",
            "pearson_spearman_strength_difference",
            "structural_category"
        ]
    ]
    .sort_values(
        "integrated_mean_rank",
        ascending=True
    )
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("Final Predictor Screening Evidence Summary")
print("=" * 80)

print(
    "Predictors assessed:",
    len(predictor_screening_evidence)
)

print(
    "Candidate redundancy members:",
    predictor_screening_evidence[
        "in_redundancy_group"
    ].sum()
)

print(
    "Special-relationship variables:",
    predictor_screening_evidence[
        "in_special_relationship"
    ].sum()
)

display(
    predictor_screening_evidence.round(4)
)

Final Predictor Screening Evidence Summary
Predictors assessed: 14
Candidate redundancy members: 3
Special-relationship variables: 7


,pearson_with_voltage,spearman_with_voltage,mutual_information,mean_abs_voltage_association,integrated_mean_rank,in_redundancy_group,strong_pair_count,in_special_relationship,special_pair_count,pearson_spearman_strength_difference,structural_category
variable,,,,,,,,,,,
current,-0.9702,-0.9284,1.5505,0.9493,1.0000,True,2,False,0,-0.0417,Candidate redundancy
total_cathode_stack_flow,-0.9222,-0.9276,1.4545,0.9249,2.3333,True,2,True,1,0.0054,Redundancy + special structure
total_anode_stack_flow,-0.9218,-0.9281,1.4411,0.9249,2.6667,True,2,True,1,0.0063,Redundancy + special structure
pressure_cathode_inlet,-0.6703,-0.7612,0.8286,0.7158,4.6667,False,0,True,1,0.0909,Special relationship structure
temp_cathode_inlet,-0.7423,-0.6558,0.7614,0.6990,5.0000,False,0,False,0,-0.0864,No structural flag
temp_anode_endplate,-0.4041,-0.4528,0.9053,0.4284,5.3333,False,0,False,0,0.0487,No structural flag
temp_anode_inlet,-0.3859,-0.3709,0.5244,0.3784,7.6667,False,0,False,0,-0.0150,No structural flag
pressure_cathode_outlet,0.3024,0.2239,0.6866,0.2631,8.3333,False,0,True,1,-0.0786,Special relationship structure
pressure_anode_inlet,-0.1128,-0.3904,0.2262,0.2516,10.3333,False,0,True,4,0.2776,Special relationship structure


In [ ]:
### Feature Engineering and Feature-Selection Implications

The preceding association analyses identified differences in predictor relevance, redundancy, relationship structure, and stability across durability stages.

This section integrates that evidence to identify implications for subsequent feature engineering and feature selection. The objective is not to remove variables or create new features at this stage, but to flag predictors that may require redundancy assessment, preservation, transformation, or further degradation-focused investigation before machine-learning modelling.

In [101]:
# ============================================================
# Feature Engineering / Selection Implications
# Step 1: Build Predictor-Level Evidence Base
# ============================================================

# Start from the final predictor screening evidence
# created in the preceding redundancy/special-variable section
feature_implications = predictor_screening_evidence.copy()


# ------------------------------------------------------------
# Add stage-wise Pearson stability evidence
# ------------------------------------------------------------

pearson_stage_evidence = (
    pearson_stability_summary[
        [
            "mean_abs_r",
            "sd_r",
            "r_range"
        ]
    ]
    .rename(columns={
        "mean_abs_r": "stagewise_pearson_mean_strength",
        "sd_r": "stagewise_pearson_sd",
        "r_range": "stagewise_pearson_range"
    })
)

feature_implications = feature_implications.join(
    pearson_stage_evidence,
    how="left"
)


# ------------------------------------------------------------
# Add stage-wise Spearman stability evidence
# ------------------------------------------------------------

spearman_stage_evidence = (
    spearman_stability_summary[
        [
            "mean_abs_rho",
            "sd_rho",
            "rho_range"
        ]
    ]
    .rename(columns={
        "mean_abs_rho": "stagewise_spearman_mean_strength",
        "sd_rho": "stagewise_spearman_sd",
        "rho_range": "stagewise_spearman_range"
    })
)

feature_implications = feature_implications.join(
    spearman_stage_evidence,
    how="left"
)


# ------------------------------------------------------------
# Count available stage-wise evidence
# ------------------------------------------------------------

n_predictors = len(feature_implications)

n_stagewise_pearson = (
    feature_implications[
        "stagewise_pearson_mean_strength"
    ]
    .notna()
    .sum()
)

n_stagewise_spearman = (
    feature_implications[
        "stagewise_spearman_mean_strength"
    ]
    .notna()
    .sum()
)


# ------------------------------------------------------------
# Display combined evidence
# ------------------------------------------------------------

print("=" * 80)
print("Predictor-Level Evidence for Feature Engineering / Selection")
print("=" * 80)

print(
    f"Predictors assessed: "
    f"{n_predictors}"
)

print(
    f"Predictors with stage-wise Pearson evidence: "
    f"{n_stagewise_pearson}"
)

print(
    f"Predictors with stage-wise Spearman evidence: "
    f"{n_stagewise_spearman}"
)

display(
    feature_implications.round(4)
)

Predictor-Level Evidence for Feature Engineering / Selection
Predictors assessed: 14
Predictors with stage-wise Pearson evidence: 6
Predictors with stage-wise Spearman evidence: 6


,pearson_with_voltage,spearman_with_voltage,mutual_information,mean_abs_voltage_association,integrated_mean_rank,in_redundancy_group,strong_pair_count,in_special_relationship,special_pair_count,pearson_spearman_strength_difference,structural_category,stagewise_pearson_mean_strength,stagewise_pearson_sd,stagewise_pearson_range,stagewise_spearman_mean_strength,stagewise_spearman_sd,stagewise_spearman_range
variable,,,,,,,,,,,,,,,,,
current,-0.9702,-0.9284,1.5505,0.9493,1.0000,True,2,False,0,-0.0417,Candidate redundancy,0.9773,0.0041,0.0134,0.9362,0.0071,0.0317
total_cathode_stack_flow,-0.9222,-0.9276,1.4545,0.9249,2.3333,True,2,True,1,0.0054,Redundancy + special structure,0.9277,0.0098,0.0428,0.9175,0.0175,0.0839
total_anode_stack_flow,-0.9218,-0.9281,1.4411,0.9249,2.6667,True,2,True,1,0.0063,Redundancy + special structure,NaN,NaN,NaN,NaN,NaN,NaN
pressure_cathode_inlet,-0.6703,-0.7612,0.8286,0.7158,4.6667,False,0,True,1,0.0909,Special relationship structure,0.6757,0.0102,0.0457,0.7713,0.0182,0.0757
temp_cathode_inlet,-0.7423,-0.6558,0.7614,0.6990,5.0000,False,0,False,0,-0.0864,No structural flag,0.7516,0.0117,0.0420,0.6502,0.0153,0.0673
temp_anode_endplate,-0.4041,-0.4528,0.9053,0.4284,5.3333,False,0,False,0,0.0487,No structural flag,0.7081,0.0408,0.1610,0.5882,0.0585,0.2518
temp_anode_inlet,-0.3859,-0.3709,0.5244,0.3784,7.6667,False,0,False,0,-0.0150,No structural flag,NaN,NaN,NaN,NaN,NaN,NaN
pressure_cathode_outlet,0.3024,0.2239,0.6866,0.2631,8.3333,False,0,True,1,-0.0786,Special relationship structure,NaN,NaN,NaN,NaN,NaN,NaN
pressure_anode_inlet,-0.1128,-0.3904,0.2262,0.2516,10.3333,False,0,True,4,0.2776,Special relationship structure,0.3197,0.1420,0.4408,0.4023,0.0655,0.2264


In [ ]:
### Predictor-Level Implication Flags

The integrated evidence is now used to identify predictors requiring particular attention during subsequent feature engineering and feature selection.

These flags are diagnostic rather than final decisions. A flagged predictor is not automatically retained, removed, or transformed. Instead, the flags indicate whether the predictor exhibits redundancy, unusual relationship structure, or stage-dependent association behaviour that should be considered during later feature development and modelling.

In [102]:
# ============================================================
# Feature Engineering / Selection Implications
# Step 2: Create Evidence-Based Predictor Flags
# ============================================================

# ------------------------------------------------------------
# 1. Redundancy flag
# ------------------------------------------------------------

feature_implications["redundancy_flag"] = (
    feature_implications["in_redundancy_group"]
)


# ------------------------------------------------------------
# 2. Special relationship flag
# ------------------------------------------------------------

feature_implications["special_relationship_flag"] = (
    feature_implications["in_special_relationship"]
)


# ------------------------------------------------------------
# 3. Stage-wise evidence availability
# ------------------------------------------------------------

feature_implications["stagewise_evidence_available"] = (
    feature_implications[
        "stagewise_pearson_mean_strength"
    ].notna()
    &
    feature_implications[
        "stagewise_spearman_mean_strength"
    ].notna()
)


# ------------------------------------------------------------
# 4. Pearson-Spearman stage-stability disagreement
#
# This is a descriptive quantity rather than a threshold-based
# classification. It shows whether Pearson and Spearman give
# different pictures of stage-wise variability.
# ------------------------------------------------------------

feature_implications[
    "stagewise_range_difference"
] = (
    feature_implications["stagewise_spearman_range"]
    -
    feature_implications["stagewise_pearson_range"]
)


# ------------------------------------------------------------
# Display diagnostic flags
# ------------------------------------------------------------

implication_flags = feature_implications[
    [
        "in_redundancy_group",
        "redundancy_flag",
        "strong_pair_count",
        "in_special_relationship",
        "special_relationship_flag",
        "special_pair_count",
        "stagewise_evidence_available",
        "stagewise_pearson_range",
        "stagewise_spearman_range",
        "stagewise_range_difference"
    ]
].copy()


print("=" * 80)
print("Predictor-Level Feature Engineering / Selection Flags")
print("=" * 80)

print(
    "Predictors flagged for candidate redundancy:",
    implication_flags["redundancy_flag"].sum()
)

print(
    "Predictors flagged for special relationship structure:",
    implication_flags["special_relationship_flag"].sum()
)

print(
    "Predictors with stage-wise stability evidence:",
    implication_flags["stagewise_evidence_available"].sum()
)

display(
    implication_flags.round(4)
)

Predictor-Level Feature Engineering / Selection Flags
Predictors flagged for candidate redundancy: 3
Predictors flagged for special relationship structure: 7
Predictors with stage-wise stability evidence: 6


,in_redundancy_group,redundancy_flag,strong_pair_count,in_special_relationship,special_relationship_flag,special_pair_count,stagewise_evidence_available,stagewise_pearson_range,stagewise_spearman_range,stagewise_range_difference
variable,,,,,,,,,,
current,True,True,2,False,False,0,True,0.0134,0.0317,0.0183
total_cathode_stack_flow,True,True,2,True,True,1,True,0.0428,0.0839,0.0411
total_anode_stack_flow,True,True,2,True,True,1,False,NaN,NaN,NaN
pressure_cathode_inlet,False,False,0,True,True,1,True,0.0457,0.0757,0.0300
temp_cathode_inlet,False,False,0,False,False,0,True,0.0420,0.0673,0.0253
temp_anode_endplate,False,False,0,False,False,0,True,0.1610,0.2518,0.0908
temp_anode_inlet,False,False,0,False,False,0,False,NaN,NaN,NaN
pressure_cathode_outlet,False,False,0,True,True,1,False,NaN,NaN,NaN
pressure_anode_inlet,False,False,0,True,True,4,True,0.4408,0.2264,-0.2144


In [ ]:
### Predictor-Level Future Action Categories

Translate the statistical evidence into practical recommendations for later feature engineering and feature selection.

The categories indicate what should be investigated in subsequent notebooks. They do not remove, retain, or transform any predictor at this stage.

In [103]:
# ============================================================
# Feature Engineering / Selection Implications
# Step 3: Assign Future Investigation Categories
# ============================================================

feature_action_summary = feature_implications.copy()


# ------------------------------------------------------------
# Create evidence-based future action recommendations
# ------------------------------------------------------------

def assign_future_actions(row):

    actions = []

    # Candidate redundancy
    if row["redundancy_flag"]:
        actions.append(
            "Assess redundancy before feature selection"
        )

    # Unusual predictor-to-predictor relationship structure
    if row["special_relationship_flag"]:
        actions.append(
            "Investigate relationship structure before transformation"
        )

    # Stage-wise stability evidence exists
    if row["stagewise_evidence_available"]:
        actions.append(
            "Consider stage-wise stability during degradation assessment"
        )

    # No specific structural/stability flag
    if len(actions) == 0:
        actions.append(
            "Retain for general feature evaluation"
        )

    return " | ".join(actions)


feature_action_summary["future_action"] = (
    feature_action_summary.apply(
        assign_future_actions,
        axis=1
    )
)


# ------------------------------------------------------------
# Create broader investigation category
# ------------------------------------------------------------

def assign_investigation_category(row):

    if (
        row["redundancy_flag"]
        and row["special_relationship_flag"]
    ):
        return "Redundancy + structural investigation"

    elif row["redundancy_flag"]:
        return "Redundancy investigation"

    elif row["special_relationship_flag"]:
        return "Structural relationship investigation"

    elif row["stagewise_evidence_available"]:
        return "Stage-wise degradation investigation"

    else:
        return "General feature evaluation"


feature_action_summary["investigation_category"] = (
    feature_action_summary.apply(
        assign_investigation_category,
        axis=1
    )
)


# ------------------------------------------------------------
# Compact output table
# ------------------------------------------------------------

feature_action_table = (
    feature_action_summary[
        [
            "integrated_mean_rank",
            "mean_abs_voltage_association",
            "mutual_information",
            "redundancy_flag",
            "special_relationship_flag",
            "stagewise_evidence_available",
            "stagewise_pearson_range",
            "stagewise_spearman_range",
            "investigation_category",
            "future_action"
        ]
    ]
    .sort_values(
        "integrated_mean_rank",
        ascending=True
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 85)
print("Future Feature Engineering / Selection Investigation Categories")
print("=" * 85)

print(
    f"Predictors assessed: "
    f"{len(feature_action_table)}"
)

print("\nInvestigation category counts:")

display(
    feature_action_table[
        "investigation_category"
    ]
    .value_counts()
    .rename("number_of_predictors")
    .to_frame()
)

print("\nPredictor-level recommendations:")

display(
    feature_action_table.round(4)
)

Future Feature Engineering / Selection Investigation Categories
Predictors assessed: 14

Investigation category counts:


,number_of_predictors
investigation_category,
Structural relationship investigation,5
General feature evaluation,4
Redundancy + structural investigation,2
Stage-wise degradation investigation,2
Redundancy investigation,1



Predictor-level recommendations:


,integrated_mean_rank,mean_abs_voltage_association,mutual_information,redundancy_flag,special_relationship_flag,stagewise_evidence_available,stagewise_pearson_range,stagewise_spearman_range,investigation_category,future_action
variable,,,,,,,,,,
current,1.0000,0.9493,1.5505,True,False,True,0.0134,0.0317,Redundancy investigation,Assess redundancy before feature selection | C...
total_cathode_stack_flow,2.3333,0.9249,1.4545,True,True,True,0.0428,0.0839,Redundancy + structural investigation,Assess redundancy before feature selection | I...
total_anode_stack_flow,2.6667,0.9249,1.4411,True,True,False,NaN,NaN,Redundancy + structural investigation,Assess redundancy before feature selection | I...
pressure_cathode_inlet,4.6667,0.7158,0.8286,False,True,True,0.0457,0.0757,Structural relationship investigation,Investigate relationship structure before tran...
temp_cathode_inlet,5.0000,0.6990,0.7614,False,False,True,0.0420,0.0673,Stage-wise degradation investigation,Consider stage-wise stability during degradati...
temp_anode_endplate,5.3333,0.4284,0.9053,False,False,True,0.1610,0.2518,Stage-wise degradation investigation,Consider stage-wise stability during degradati...
temp_anode_inlet,7.6667,0.3784,0.5244,False,False,False,NaN,NaN,General feature evaluation,Retain for general feature evaluation
pressure_cathode_outlet,8.3333,0.2631,0.6866,False,True,False,NaN,NaN,Structural relationship investigation,Investigate relationship structure before tran...
pressure_anode_inlet,10.3333,0.2516,0.2262,False,True,True,0.4408,0.2264,Structural relationship investigation,Investigate relationship structure before tran...


In [ ]:
### Decision-Support Summary for Future Feature Engineering and Selection

The association, redundancy, special-relationship, and stage-wise stability analyses are consolidated into a predictor-level decision-support summary.

The resulting recommendations are intended to guide subsequent feature engineering and model development. They do not constitute final feature-selection decisions. Predictors are therefore not removed, transformed, or combined in this notebook solely on the basis of association analysis.

In [104]:
# ============================================================
# Feature Engineering / Selection Implications
# Step 4: Final Decision-Support Summary
# ============================================================

feature_decision_summary = feature_action_summary.copy()


# ------------------------------------------------------------
# Assign decision-support recommendation
# ------------------------------------------------------------

def assign_decision_support(row):

    # Redundant AND structurally unusual
    if (
        row["redundancy_flag"]
        and row["special_relationship_flag"]
    ):
        return (
            "Preserve for now; assess redundancy and "
            "relationship structure before modelling"
        )

    # Redundancy only
    elif row["redundancy_flag"]:
        return (
            "Preserve for now; compare with correlated "
            "predictors during feature selection"
        )

    # Special relationship + stage-wise evidence
    elif (
        row["special_relationship_flag"]
        and row["stagewise_evidence_available"]
    ):
        return (
            "Preserve for now; investigate structural and "
            "stage-dependent behaviour"
        )

    # Special relationship only
    elif row["special_relationship_flag"]:
        return (
            "Preserve for now; investigate relationship "
            "structure and possible transformation"
        )

    # Stage-wise evidence only
    elif row["stagewise_evidence_available"]:
        return (
            "Preserve for now; evaluate stage-wise behaviour "
            "for degradation relevance"
        )

    # No major structural flag
    else:
        return (
            "Preserve for now; evaluate predictive contribution "
            "during modelling"
        )


feature_decision_summary["decision_support"] = (
    feature_decision_summary.apply(
        assign_decision_support,
        axis=1
    )
)


# ------------------------------------------------------------
# Create compact final table
# ------------------------------------------------------------

final_feature_decision_table = (
    feature_decision_summary[
        [
            "integrated_mean_rank",
            "mean_abs_voltage_association",
            "mutual_information",
            "redundancy_flag",
            "special_relationship_flag",
            "stagewise_evidence_available",
            "investigation_category",
            "decision_support"
        ]
    ]
    .sort_values(
        "integrated_mean_rank",
        ascending=True
    )
)


# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("=" * 90)
print("Final Feature Engineering / Selection Decision-Support Summary")
print("=" * 90)

print(f"Predictors assessed: {len(final_feature_decision_table)}")

print(
    "Predictors removed at this stage: 0"
)

print(
    "Predictors retained for subsequent investigation:",
    len(final_feature_decision_table)
)

print("\nDecision-support recommendations:")

display(
    final_feature_decision_table.round(4)
)

Final Feature Engineering / Selection Decision-Support Summary
Predictors assessed: 14
Predictors removed at this stage: 0
Predictors retained for subsequent investigation: 14

Decision-support recommendations:


,integrated_mean_rank,mean_abs_voltage_association,mutual_information,redundancy_flag,special_relationship_flag,stagewise_evidence_available,investigation_category,decision_support
variable,,,,,,,,
current,1.0000,0.9493,1.5505,True,False,True,Redundancy investigation,Preserve for now; compare with correlated pred...
total_cathode_stack_flow,2.3333,0.9249,1.4545,True,True,True,Redundancy + structural investigation,Preserve for now; assess redundancy and relati...
total_anode_stack_flow,2.6667,0.9249,1.4411,True,True,False,Redundancy + structural investigation,Preserve for now; assess redundancy and relati...
pressure_cathode_inlet,4.6667,0.7158,0.8286,False,True,True,Structural relationship investigation,Preserve for now; investigate structural and s...
temp_cathode_inlet,5.0000,0.6990,0.7614,False,False,True,Stage-wise degradation investigation,Preserve for now; evaluate stage-wise behaviou...
temp_anode_endplate,5.3333,0.4284,0.9053,False,False,True,Stage-wise degradation investigation,Preserve for now; evaluate stage-wise behaviou...
temp_anode_inlet,7.6667,0.3784,0.5244,False,False,False,General feature evaluation,Preserve for now; evaluate predictive contribu...
pressure_cathode_outlet,8.3333,0.2631,0.6866,False,True,False,Structural relationship investigation,Preserve for now; investigate relationship str...
pressure_anode_inlet,10.3333,0.2516,0.2262,False,True,True,Structural relationship investigation,Preserve for now; investigate structural and s...


In [ ]:
### Scientific Interpretation of Association Structure

Statistical association does not by itself identify the physical mechanism responsible for a relationship.

The association results are therefore interpreted in the context of PEMFC operation. Relationships are considered in terms of whether they may primarily reflect operating-load dependence, control-system behaviour, physical coupling between subsystem variables, mathematical dependence, or potentially degradation-sensitive behaviour.

These categories are interpretive rather than causal. Correlation, rank association, and mutual information alone cannot establish that one operational variable causes changes in voltage or that a relationship is directly caused by degradation.

In [ ]:
#### Scientific Interpretation Categories

The following categories are used to organise the statistical findings:

- **Load-driven:** relationships primarily associated with changes in electrical load and the corresponding operating response of the fuel-cell system.

- **Control-driven:** relationships that may arise because operational variables are regulated or adjusted by the test/control system in response to operating conditions.

- **Physically coupled:** relationships expected because variables belong to connected physical or thermodynamic processes within the PEMFC system.

- **Mathematically dependent:** relationships arising wholly or partly from direct mathematical dependence between variables rather than independent physical information.

- **Potentially degradation-sensitive:** relationships whose behaviour across durability stages may provide information relevant to changing PEMFC condition. Stage dependence alone is not treated as proof of degradation.

These categories are not mutually exclusive. A variable or relationship may belong to more than one category.

In [ ]:
#### Predictor-Level Evidence for Scientific Interpretation

The statistical evidence obtained from the association analysis is consolidated before assigning physical interpretations.

For each operational predictor, the table combines its association with voltage, nonlinear dependence information, redundancy or special-relationship structure, and available stage-wise stability evidence.

This provides a transparent evidence base for distinguishing operational coupling from potentially degradation-sensitive behaviour. Scientific categories are assigned subsequently using PEMFC knowledge rather than statistical thresholds alone.

In [105]:
# ============================================================
# Scientific Interpretation
# Step 2: Build Predictor-Level Interpretation Evidence
# ============================================================

scientific_interpretation = feature_decision_summary.copy()


# ------------------------------------------------------------
# Select the evidence required for scientific interpretation
# ------------------------------------------------------------

scientific_evidence_table = scientific_interpretation[
    [
        "pearson_with_voltage",
        "spearman_with_voltage",
        "mutual_information",
        "mean_abs_voltage_association",
        "integrated_mean_rank",
        "redundancy_flag",
        "special_relationship_flag",
        "stagewise_evidence_available",
        "stagewise_pearson_range",
        "stagewise_spearman_range"
    ]
].copy()


# ------------------------------------------------------------
# Add Pearson-Spearman difference for interpretation
# ------------------------------------------------------------

scientific_evidence_table[
    "pearson_spearman_difference"
] = (
    scientific_evidence_table["spearman_with_voltage"].abs()
    - scientific_evidence_table["pearson_with_voltage"].abs()
)


# ------------------------------------------------------------
# Sort using existing integrated association ranking
# ------------------------------------------------------------

scientific_evidence_table = (
    scientific_evidence_table
    .sort_values(
        "integrated_mean_rank",
        ascending=True
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 85)
print("Predictor-Level Evidence for Scientific PEMFC Interpretation")
print("=" * 85)

print(
    f"Predictors assessed: "
    f"{len(scientific_evidence_table)}"
)

print(
    "Stage-wise evidence available for:",
    int(
        scientific_evidence_table[
            "stagewise_evidence_available"
        ].sum()
    ),
    "predictors"
)

print("\nScientific interpretation evidence:")

display(
    scientific_evidence_table.round(4)
)

Predictor-Level Evidence for Scientific PEMFC Interpretation
Predictors assessed: 14
Stage-wise evidence available for: 6 predictors

Scientific interpretation evidence:


,pearson_with_voltage,spearman_with_voltage,mutual_information,mean_abs_voltage_association,integrated_mean_rank,redundancy_flag,special_relationship_flag,stagewise_evidence_available,stagewise_pearson_range,stagewise_spearman_range,pearson_spearman_difference
variable,,,,,,,,,,,
current,-0.9702,-0.9284,1.5505,0.9493,1.0000,True,False,True,0.0134,0.0317,-0.0417
total_cathode_stack_flow,-0.9222,-0.9276,1.4545,0.9249,2.3333,True,True,True,0.0428,0.0839,0.0054
total_anode_stack_flow,-0.9218,-0.9281,1.4411,0.9249,2.6667,True,True,False,NaN,NaN,0.0063
pressure_cathode_inlet,-0.6703,-0.7612,0.8286,0.7158,4.6667,False,True,True,0.0457,0.0757,0.0909
temp_cathode_inlet,-0.7423,-0.6558,0.7614,0.6990,5.0000,False,False,True,0.0420,0.0673,-0.0864
temp_anode_endplate,-0.4041,-0.4528,0.9053,0.4284,5.3333,False,False,True,0.1610,0.2518,0.0487
temp_anode_inlet,-0.3859,-0.3709,0.5244,0.3784,7.6667,False,False,False,NaN,NaN,-0.0150
pressure_cathode_outlet,0.3024,0.2239,0.6866,0.2631,8.3333,False,True,False,NaN,NaN,-0.0786
pressure_anode_inlet,-0.1128,-0.3904,0.2262,0.2516,10.3333,False,True,True,0.4408,0.2264,0.2776


In [ ]:
#### Physical and Operational Context of Predictors

Each predictor is assigned an initial physical or operational context based on its role within the PEMFC system.

This classification is not inferred automatically from correlation magnitude. Instead, it provides the engineering context required to interpret the statistical associations.

The purpose is to distinguish variables that primarily represent electrical loading, reactant supply, pressure behaviour, thermal behaviour, and humidification/water-management conditions before assessing whether their statistical behaviour may contain degradation-relevant information.

In [106]:
# ============================================================
# Scientific Interpretation
# Step 3: Assign Physical / Operational Context
# ============================================================

scientific_classification = scientific_evidence_table.copy()


# ------------------------------------------------------------
# Define PEMFC physical / operational context
# ------------------------------------------------------------

physical_context = {

    "current":
        "Electrical load",

    "total_anode_stack_flow":
        "Anode reactant supply",

    "total_cathode_stack_flow":
        "Cathode reactant supply",

    "pressure_anode_inlet":
        "Anode pressure / reactant delivery",

    "pressure_anode_outlet":
        "Anode pressure / reactant transport",

    "pressure_cathode_inlet":
        "Cathode pressure / reactant delivery",

    "pressure_cathode_outlet":
        "Cathode pressure / reactant transport",

    "temp_anode_endplate":
        "Stack thermal condition",

    "temp_anode_inlet":
        "Anode thermal condition",

    "temp_anode_outlet":
        "Anode thermal condition",

    "temp_cathode_inlet":
        "Cathode thermal condition",

    "temp_cathode_outlet":
        "Cathode thermal condition",

    "temp_anode_dewpoint_water":
        "Anode humidification / water management",

    "temp_cathode_dewpoint_water":
        "Cathode humidification / water management"
}


# ------------------------------------------------------------
# Map physical context to predictors
# ------------------------------------------------------------

scientific_classification["physical_context"] = (
    scientific_classification.index.map(
        physical_context
    )
)


# ------------------------------------------------------------
# Verify that every predictor received a classification
# ------------------------------------------------------------

unclassified_variables = (
    scientific_classification[
        scientific_classification["physical_context"].isna()
    ].index.tolist()
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("PEMFC Physical / Operational Context of Predictors")
print("=" * 80)

print(
    f"Predictors classified: "
    f"{scientific_classification['physical_context'].notna().sum()}"
)

print(
    f"Predictors without classification: "
    f"{len(unclassified_variables)}"
)

if unclassified_variables:
    print(
        "\nUnclassified predictors:",
        unclassified_variables
    )

display(
    scientific_classification[
        [
            "physical_context",
            "pearson_with_voltage",
            "spearman_with_voltage",
            "mutual_information",
            "integrated_mean_rank"
        ]
    ].round(4)
)

PEMFC Physical / Operational Context of Predictors
Predictors classified: 14
Predictors without classification: 0


,physical_context,pearson_with_voltage,spearman_with_voltage,mutual_information,integrated_mean_rank
variable,,,,,
current,Electrical load,-0.9702,-0.9284,1.5505,1.0000
total_cathode_stack_flow,Cathode reactant supply,-0.9222,-0.9276,1.4545,2.3333
total_anode_stack_flow,Anode reactant supply,-0.9218,-0.9281,1.4411,2.6667
pressure_cathode_inlet,Cathode pressure / reactant delivery,-0.6703,-0.7612,0.8286,4.6667
temp_cathode_inlet,Cathode thermal condition,-0.7423,-0.6558,0.7614,5.0000
temp_anode_endplate,Stack thermal condition,-0.4041,-0.4528,0.9053,5.3333
temp_anode_inlet,Anode thermal condition,-0.3859,-0.3709,0.5244,7.6667
pressure_cathode_outlet,Cathode pressure / reactant transport,0.3024,0.2239,0.6866,8.3333
pressure_anode_inlet,Anode pressure / reactant delivery,-0.1128,-0.3904,0.2262,10.3333


In [ ]:
#### Scientific Relationship Classification

The predictor-level evidence is now interpreted in terms of the mechanisms that may generate the observed associations.

Multiple interpretations may apply to the same predictor. For example, reactant flow can be load-driven because reactant demand changes with current, while simultaneously being control-driven because the test system regulates reactant supply.

Potential degradation sensitivity is treated separately and cautiously. A variable is not classified as degradation-related solely because it correlates with voltage. Instead, stage-wise changes or unusual relationship structure are used to identify variables requiring further degradation-focused investigation.

These classifications therefore represent scientific hypotheses supported by the exploratory analysis rather than causal conclusions.

In [107]:
# ============================================================
# Scientific Interpretation
# Step 4: Scientific Relationship Classification
# ============================================================

scientific_relationships = scientific_classification.copy()


# ------------------------------------------------------------
# Define scientifically motivated relationship categories
# ------------------------------------------------------------

relationship_context = {

    "current": [
        "Load-driven"
    ],

    "total_anode_stack_flow": [
        "Load-driven",
        "Control-driven",
        "Physically coupled"
    ],

    "total_cathode_stack_flow": [
        "Load-driven",
        "Control-driven",
        "Physically coupled"
    ],

    "pressure_anode_inlet": [
        "Control-driven",
        "Physically coupled",
        "Potentially degradation-sensitive"
    ],

    "pressure_anode_outlet": [
        "Physically coupled"
    ],

    "pressure_cathode_inlet": [
        "Load-driven",
        "Control-driven",
        "Physically coupled"
    ],

    "pressure_cathode_outlet": [
        "Physically coupled"
    ],

    "temp_anode_endplate": [
        "Load-driven",
        "Physically coupled",
        "Potentially degradation-sensitive"
    ],

    "temp_anode_inlet": [
        "Control-driven",
        "Physically coupled"
    ],

    "temp_anode_outlet": [
        "Physically coupled"
    ],

    "temp_cathode_inlet": [
        "Load-driven",
        "Control-driven",
        "Physically coupled"
    ],

    "temp_cathode_outlet": [
        "Physically coupled"
    ],

    "temp_anode_dewpoint_water": [
        "Control-driven",
        "Physically coupled"
    ],

    "temp_cathode_dewpoint_water": [
        "Control-driven",
        "Physically coupled"
    ]
}


# ------------------------------------------------------------
# Add categories
# ------------------------------------------------------------

scientific_relationships["scientific_relationship"] = (
    scientific_relationships.index.map(
        lambda variable:
        " | ".join(
            relationship_context.get(variable, [])
        )
    )
)


# ------------------------------------------------------------
# Create individual Boolean columns
# ------------------------------------------------------------

categories = [
    "Load-driven",
    "Control-driven",
    "Physically coupled",
    "Potentially degradation-sensitive"
]


for category in categories:

    column_name = (
        category.lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    scientific_relationships[column_name] = (
        scientific_relationships[
            "scientific_relationship"
        ].str.contains(
            category,
            regex=False
        )
    )


# ------------------------------------------------------------
# Display compact interpretation table
# ------------------------------------------------------------

scientific_relationship_table = (
    scientific_relationships[
        [
            "physical_context",
            "scientific_relationship",
            "load_driven",
            "control_driven",
            "physically_coupled",
            "potentially_degradation_sensitive"
        ]
    ]
)


print("=" * 85)
print("Scientific Classification of Predictor Relationships")
print("=" * 85)

print(
    f"Predictors assessed: "
    f"{len(scientific_relationship_table)}"
)

print("\nCategory counts:")

category_counts = pd.Series({

    "Load-driven":
        scientific_relationships[
            "load_driven"
        ].sum(),

    "Control-driven":
        scientific_relationships[
            "control_driven"
        ].sum(),

    "Physically coupled":
        scientific_relationships[
            "physically_coupled"
        ].sum(),

    "Potentially degradation-sensitive":
        scientific_relationships[
            "potentially_degradation_sensitive"
        ].sum()
})

display(
    category_counts
    .rename("number_of_predictors")
    .to_frame()
)

print("\nPredictor-level scientific classification:")

display(
    scientific_relationship_table
)

Scientific Classification of Predictor Relationships
Predictors assessed: 14

Category counts:


,number_of_predictors
Load-driven,6
Control-driven,8
Physically coupled,13
Potentially degradation-sensitive,2



Predictor-level scientific classification:


,physical_context,scientific_relationship,load_driven,control_driven,physically_coupled,potentially_degradation_sensitive
variable,,,,,,
current,Electrical load,Load-driven,True,False,False,False
total_cathode_stack_flow,Cathode reactant supply,Load-driven | Control-driven | Physically coupled,True,True,True,False
total_anode_stack_flow,Anode reactant supply,Load-driven | Control-driven | Physically coupled,True,True,True,False
pressure_cathode_inlet,Cathode pressure / reactant delivery,Load-driven | Control-driven | Physically coupled,True,True,True,False
temp_cathode_inlet,Cathode thermal condition,Load-driven | Control-driven | Physically coupled,True,True,True,False
temp_anode_endplate,Stack thermal condition,Load-driven | Physically coupled | Potentially...,True,False,True,True
temp_anode_inlet,Anode thermal condition,Control-driven | Physically coupled,False,True,True,False
pressure_cathode_outlet,Cathode pressure / reactant transport,Physically coupled,False,False,True,False
pressure_anode_inlet,Anode pressure / reactant delivery,Control-driven | Physically coupled | Potentia...,False,True,True,True


In [ ]:
#### Scientific Evidence-to-Interpretation Summary

The statistical evidence and PEMFC physical context are now integrated to provide an interpretation of the observed predictor–voltage relationships.

The purpose is to distinguish strong associations that are primarily explained by normal operating-load or control behaviour from relationships that may warrant further investigation for degradation sensitivity.

These interpretations remain exploratory. In particular, a strong association with voltage is not treated as evidence of degradation, while stage-dependent or structurally unusual behaviour is treated as a reason for further investigation rather than proof of an aging mechanism.

In [108]:
# ============================================================
# Scientific Interpretation
# Step 5: Evidence-to-Interpretation Summary
# ============================================================

scientific_summary = scientific_relationships.copy()


# ------------------------------------------------------------
# Define evidence-based interpretation notes
# ------------------------------------------------------------

interpretation_notes = {

    "current": (
        "Very strong and highly stage-stable association with voltage. "
        "Primarily represents electrical load/polarization behaviour; "
        "strong association should not be interpreted as direct "
        "degradation sensitivity."
    ),

    "total_cathode_stack_flow": (
        "Very strong association with voltage and strong redundancy with "
        "current and anode flow. Likely reflects coordinated reactant "
        "supply with operating load and control behaviour rather than "
        "independent degradation information."
    ),

    "total_anode_stack_flow": (
        "Very strong association with voltage and strong redundancy with "
        "current and cathode flow. Likely reflects coordinated reactant "
        "supply and operating control; independent predictive contribution "
        "should be assessed later."
    ),

    "pressure_cathode_inlet": (
        "Moderate-to-strong voltage association with relatively stable "
        "stage-wise behaviour. Consistent with load/control-dependent "
        "reactant delivery and physical coupling within the cathode system."
    ),

    "temp_cathode_inlet": (
        "Moderate-to-strong negative association with voltage and relatively "
        "stable stage-wise behaviour. Likely reflects thermal and operating "
        "condition coupling rather than clear degradation-specific behaviour."
    ),

    "temp_anode_endplate": (
        "Moderate global association with voltage but greater stage-wise "
        "variation than the more stable load-related predictors. This makes "
        "the variable relevant for further investigation of changing thermal "
        "behaviour across durability stages, without establishing degradation "
        "causality."
    ),

    "temp_anode_inlet": (
        "Moderate association with voltage without major redundancy or "
        "special-relationship flags. Retain as an operational thermal "
        "variable and assess its predictive contribution during modelling."
    ),

    "pressure_cathode_outlet": (
        "Relatively weak global voltage association but special predictor "
        "relationship structure. May contain subsystem-level reactant "
        "transport information not adequately represented by its direct "
        "correlation with voltage."
    ),

    "pressure_anode_inlet": (
        "Weak global Pearson association but stronger Spearman association, "
        "substantial Pearson-Spearman disagreement, unusual predictor "
        "relationship structure, and pronounced stage-wise variation. "
        "This variable warrants further investigation for condition- or "
        "degradation-sensitive behaviour, but the present analysis does "
        "not establish degradation causality."
    ),

    "temp_cathode_outlet": (
        "Weak direct association with voltage. Retain as a cathode thermal "
        "variable because direct correlation alone does not determine its "
        "future predictive usefulness."
    ),

    "pressure_anode_outlet": (
        "Weak direct voltage association with special relationship structure "
        "relative to other predictors. This suggests subsystem behaviour "
        "that should be examined through physically meaningful pressure "
        "features and later modelling."
    ),

    "temp_anode_outlet": (
        "Very weak direct voltage association but special relationship "
        "structure. Its usefulness may emerge through thermal gradients or "
        "interaction features rather than as an isolated raw predictor."
    ),

    "temp_cathode_dewpoint_water": (
        "Very weak direct linear and monotonic association with voltage, "
        "although nonlinear information is present. Retain for later "
        "assessment of humidification and water-management features."
    ),

    "temp_anode_dewpoint_water": (
        "Very weak direct linear and monotonic association with voltage, "
        "although nonlinear information is present. Retain for later "
        "assessment of humidification and water-management features."
    )
}


# ------------------------------------------------------------
# Map interpretation notes
# ------------------------------------------------------------

scientific_summary["scientific_interpretation"] = (
    scientific_summary.index.map(
        interpretation_notes
    )
)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

missing_interpretations = (
    scientific_summary[
        scientific_summary["scientific_interpretation"].isna()
    ].index.tolist()
)


print("=" * 90)
print("Scientific Evidence-to-Interpretation Summary")
print("=" * 90)

print(
    f"Predictors interpreted: "
    f"{scientific_summary['scientific_interpretation'].notna().sum()}"
)

print(
    f"Predictors without interpretation: "
    f"{len(missing_interpretations)}"
)

if missing_interpretations:
    print(
        "\nPredictors requiring interpretation:",
        missing_interpretations
    )


# ------------------------------------------------------------
# Compact final scientific interpretation table
# ------------------------------------------------------------

scientific_interpretation_summary = (
    scientific_summary[
        [
            "physical_context",
            "pearson_with_voltage",
            "spearman_with_voltage",
            "mutual_information",
            "scientific_relationship",
            "scientific_interpretation"
        ]
    ]
    .sort_values(
        "mutual_information",
        ascending=False
    )
)


display(
    scientific_interpretation_summary.round(4)
)

Scientific Evidence-to-Interpretation Summary
Predictors interpreted: 14
Predictors without interpretation: 0


,physical_context,pearson_with_voltage,spearman_with_voltage,mutual_information,scientific_relationship,scientific_interpretation
variable,,,,,,
current,Electrical load,-0.9702,-0.9284,1.5505,Load-driven,Very strong and highly stage-stable associatio...
total_cathode_stack_flow,Cathode reactant supply,-0.9222,-0.9276,1.4545,Load-driven | Control-driven | Physically coupled,Very strong association with voltage and stron...
total_anode_stack_flow,Anode reactant supply,-0.9218,-0.9281,1.4411,Load-driven | Control-driven | Physically coupled,Very strong association with voltage and stron...
temp_anode_endplate,Stack thermal condition,-0.4041,-0.4528,0.9053,Load-driven | Physically coupled | Potentially...,Moderate global association with voltage but g...
pressure_cathode_inlet,Cathode pressure / reactant delivery,-0.6703,-0.7612,0.8286,Load-driven | Control-driven | Physically coupled,Moderate-to-strong voltage association with re...
temp_cathode_inlet,Cathode thermal condition,-0.7423,-0.6558,0.7614,Load-driven | Control-driven | Physically coupled,Moderate-to-strong negative association with v...
pressure_cathode_outlet,Cathode pressure / reactant transport,0.3024,0.2239,0.6866,Physically coupled,Relatively weak global voltage association but...
temp_anode_inlet,Anode thermal condition,-0.3859,-0.3709,0.5244,Control-driven | Physically coupled,Moderate association with voltage without majo...
temp_anode_outlet,Anode thermal condition,-0.0399,-0.0611,0.3541,Physically coupled,Very weak direct voltage association but speci...


In [ ]:
### Final Notebook 9 Association Analysis Summary

The association analysis combined Pearson correlation, Spearman correlation, Mutual Information, visual relationship assessment, stage-wise stability analysis, predictor redundancy assessment, and PEMFC scientific interpretation.

The objective was not simply to rank predictors by correlation with voltage, but to determine whether their relationships were linear, monotonic, nonlinear, stable across durability stages, redundant with other predictors, or potentially informative for later degradation modelling.

No predictors are removed in this notebook. The results are used to guide subsequent feature engineering, feature selection, degradation analysis, and machine-learning model development.

In [109]:
# ============================================================
# Notebook 9 — Final Association Analysis Summary
# ============================================================

notebook9_summary = pd.DataFrame({

    "analysis_component": [
        "Pearson correlation",
        "Spearman correlation",
        "Mutual Information",
        "Integrated association ranking",
        "Visual relationship assessment",
        "Stage-wise Pearson stability",
        "Stage-wise Spearman stability",
        "Predictor redundancy assessment",
        "Special relationship assessment",
        "Feature engineering / selection implications",
        "Scientific PEMFC interpretation"
    ],

    "purpose": [
        "Assess linear predictor-voltage relationships",
        "Assess monotonic predictor-voltage relationships",
        "Detect broader and nonlinear statistical dependence",
        "Compare predictor importance across association methods",
        "Inspect relationship structure and Pearson linearity assumptions",
        "Determine whether linear associations remain stable across durability stages",
        "Determine whether monotonic associations remain stable across durability stages",
        "Identify strongly coupled predictors carrying overlapping information",
        "Identify predictor pairs with unusual Pearson-Spearman disagreement",
        "Translate association evidence into recommendations for later feature development",
        "Relate statistical patterns to PEMFC operating and physical mechanisms"
    ],

    "status": [
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed"
    ]
})


# ------------------------------------------------------------
# Key quantitative outcomes
# ------------------------------------------------------------

key_notebook9_outcomes = pd.DataFrame({

    "item": [
        "Candidate predictors analysed",
        "Observations used for global association analysis",
        "Durability stages represented",
        "Predictors assessed stage-wise",
        "Candidate redundancy-group members",
        "Special-relationship variables",
        "Predictors removed in Notebook 9",
        "Predictors carried forward"
    ],

    "result": [
        len(redundancy_predictors),
        f"{len(association_df):,}",
        association_df["operating_hour"].nunique(),
        int(
            scientific_evidence_table[
                "stagewise_evidence_available"
            ].sum()
        ),
        int(
            feature_implications[
                "redundancy_flag"
            ].sum()
        ),
        int(
            feature_implications[
                "special_relationship_flag"
            ].sum()
        ),
        0,
        len(redundancy_predictors)
    ]
})


# ------------------------------------------------------------
# Final predictor-level evidence
# ------------------------------------------------------------

final_notebook9_predictor_summary = (
    scientific_summary[
        [
            "physical_context",
            "pearson_with_voltage",
            "spearman_with_voltage",
            "mutual_information",
            "integrated_mean_rank",
            "redundancy_flag",
            "special_relationship_flag",
            "stagewise_evidence_available",
            "scientific_relationship",
            "scientific_interpretation"
        ]
    ]
    .sort_values(
        "integrated_mean_rank",
        ascending=True
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 90)
print("NOTEBOOK 9 — ASSOCIATION ANALYSIS COMPLETION SUMMARY")
print("=" * 90)

print("\nAnalysis components:")
display(notebook9_summary)

print("\nKey quantitative outcomes:")
display(key_notebook9_outcomes)

print("\nFinal predictor-level association evidence:")
display(
    final_notebook9_predictor_summary.round(4)
)


# ------------------------------------------------------------
# Methodological conclusion
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("Methodological Conclusion")
print("=" * 90)

print(
    "Association analysis is complete. "
    "Predictors have been evaluated using complementary linear, "
    "monotonic, nonlinear, stage-wise, redundancy, and physical "
    "interpretation perspectives."
)

print(
    "\nNo predictors were removed solely on the basis of association analysis."
)

print(
    "\nThe results will inform subsequent feature engineering, "
    "feature selection, degradation-focused analysis, and modelling."
)

print(
    "\nImportant limitation: association and stage-wise variation do not "
    "establish causality or prove that a predictor is directly driven "
    "by PEMFC degradation."
)

NOTEBOOK 9 — ASSOCIATION ANALYSIS COMPLETION SUMMARY

Analysis components:


,analysis_component,purpose,status
0,Pearson correlation,Assess linear predictor-voltage relationships,Completed
1,Spearman correlation,Assess monotonic predictor-voltage relationships,Completed
2,Mutual Information,Detect broader and nonlinear statistical depen...,Completed
3,Integrated association ranking,Compare predictor importance across associatio...,Completed
4,Visual relationship assessment,Inspect relationship structure and Pearson lin...,Completed
5,Stage-wise Pearson stability,Determine whether linear associations remain s...,Completed
6,Stage-wise Spearman stability,Determine whether monotonic associations remai...,Completed
7,Predictor redundancy assessment,Identify strongly coupled predictors carrying ...,Completed
8,Special relationship assessment,Identify predictor pairs with unusual Pearson-...,Completed
9,Feature engineering / selection implications,Translate association evidence into recommenda...,Completed



Key quantitative outcomes:


,item,result
0,Candidate predictors analysed,14
1,Observations used for global association analysis,"3,629,680"
2,Durability stages represented,20
3,Predictors assessed stage-wise,6
4,Candidate redundancy-group members,3
5,Special-relationship variables,7
6,Predictors removed in Notebook 9,0
7,Predictors carried forward,14



Final predictor-level association evidence:


,physical_context,pearson_with_voltage,spearman_with_voltage,mutual_information,integrated_mean_rank,redundancy_flag,special_relationship_flag,stagewise_evidence_available,scientific_relationship,scientific_interpretation
variable,,,,,,,,,,
current,Electrical load,-0.9702,-0.9284,1.5505,1.0000,True,False,True,Load-driven,Very strong and highly stage-stable associatio...
total_cathode_stack_flow,Cathode reactant supply,-0.9222,-0.9276,1.4545,2.3333,True,True,True,Load-driven | Control-driven | Physically coupled,Very strong association with voltage and stron...
total_anode_stack_flow,Anode reactant supply,-0.9218,-0.9281,1.4411,2.6667,True,True,False,Load-driven | Control-driven | Physically coupled,Very strong association with voltage and stron...
pressure_cathode_inlet,Cathode pressure / reactant delivery,-0.6703,-0.7612,0.8286,4.6667,False,True,True,Load-driven | Control-driven | Physically coupled,Moderate-to-strong voltage association with re...
temp_cathode_inlet,Cathode thermal condition,-0.7423,-0.6558,0.7614,5.0000,False,False,True,Load-driven | Control-driven | Physically coupled,Moderate-to-strong negative association with v...
temp_anode_endplate,Stack thermal condition,-0.4041,-0.4528,0.9053,5.3333,False,False,True,Load-driven | Physically coupled | Potentially...,Moderate global association with voltage but g...
temp_anode_inlet,Anode thermal condition,-0.3859,-0.3709,0.5244,7.6667,False,False,False,Control-driven | Physically coupled,Moderate association with voltage without majo...
pressure_cathode_outlet,Cathode pressure / reactant transport,0.3024,0.2239,0.6866,8.3333,False,True,False,Physically coupled,Relatively weak global voltage association but...
pressure_anode_inlet,Anode pressure / reactant delivery,-0.1128,-0.3904,0.2262,10.3333,False,True,True,Control-driven | Physically coupled | Potentia...,Weak global Pearson association but stronger S...



Methodological Conclusion
Association analysis is complete. Predictors have been evaluated using complementary linear, monotonic, nonlinear, stage-wise, redundancy, and physical interpretation perspectives.

No predictors were removed solely on the basis of association analysis.

The results will inform subsequent feature engineering, feature selection, degradation-focused analysis, and modelling.

Important limitation: association and stage-wise variation do not establish causality or prove that a predictor is directly driven by PEMFC degradation.
